# Non-reversible beats reversible anchored Langevin — accuracy curves only

Two figures, **training and test accuracy against iteration**, one per constraint
set. No other plots. The vertical axis is fixed to $[0.4, 0.9]$ — it is **not**
windowed on the plateau.

## Target and update

$$U(w)=\underbrace{\sum_j\bigl[\mathrm{softplus}(x_j^\top w)-y_jx_j^\top w\bigr]+\frac{w_0^2}{2\sigma^2}}_{f,\ \text{differentiable}}
\;+\;\underbrace{\lambda_{\mathrm{lasso}}\sum_{j\ge1}|w_j|}_{g,\ \text{NOT differentiable}}$$

The anchor $U_0=f+g_\delta$ replaces $|w_j|$ by $\sqrt{w_j^2+\delta^2}$ and
$a(w)=e^{U(w)-U_0(w)}\in[e^{-8\lambda\delta},1]$. Both methods use the **exact** gradient:

$$w_{k+1}=\Pi_K\!\Bigl(w_k-\eta\,a(w_k)\nabla U_0(w_k)+\eta\,\alpha\,a(w_k)\,J_s(w_k)\nabla U_0(w_k)+\sqrt{2\eta\,a(w_k)}\,\xi_{k+1}\Bigr)$$

$\alpha=0$ is the reversible method, $\alpha=1$ the non-reversible one. Within a
replicate the two chains share the initial point and every Gaussian increment; the
only difference is $\alpha$. $J_s$ is block diagonal on the coordinate triples
$(0,1,2),(3,4,5),(6,7,8)$, block $I$ being the cross-product matrix $[v_I]_\times$ with
$v_I=s\,w_I$ (ball) or $v_I=-s\nabla_I g_\varepsilon(w)$ (smoothed $L_1$ ball).

## Why this design, in one paragraph

$[v_I]_\times$ rotates only in the plane **perpendicular to its axis** $v_I$. Linearising
the drift $-\eta a(I-J)H$ at the mode, a slow Hessian eigen-direction $q_3$ that lies in
that plane, with fast partner $q_2$, has its rate replaced by $(\lambda_2+\lambda_3)/2$
once $\sigma=s|v_I|\ge\sigma^*=(\lambda_2-\lambda_3)/(2\sqrt{\lambda_2\lambda_3})$ — a
speed-up of $(\kappa+1)/2$, $\kappa=\lambda_2/\lambda_3$ — while a slow direction *along*
the axis is untouched and an oblique one is capped at $\sim1/\cos^2\theta$. In discrete
time the rotation is stable while $\sigma^2<(\lambda_a+\lambda_b)/(\eta a\lambda_a\lambda_b)-1$
for the pair spanning the rotated plane. Under the $L_1$ geometry the axis is the
soft-sign of $w_I$, i.e. $\approx(1,1,1)/\sqrt3$ when the three coefficients are positive.
So the design ("unstandardised covariates") gives each slope triple the covariance
$v_{\rm axis}q_1q_1^\top+v_{\rm fast}q_2q_2^\top+v_{\rm slow}q_3q_3^\top$ with
$q_1=(1,1,1)/\sqrt3$, $q_2=(0,1,-1)/\sqrt2$ (fast, $q_2\cdot\beta=0$: no signal) and
$q_3=(2,-1,-1)/\sqrt6$ (slow, in the rotated plane, and carrying signal because
$\beta_I=(b,e,e)$ with $b>e$). The reversible chain needs $\sim1/(\eta a\lambda_3)$
iterations along $q_3$; the non-reversible one needs $\sim2/(\eta a\lambda_2)$. Because the
$q_3$ residual carries an $O(1)$ share of the logit variance, the accuracy gap during
that transient is large. It is a **convergence-speed** effect: both chains reach the same
plateau, and the gap closes once the reversible chain has converged.

Single-iterate accuracy: at checkpoint $k$ every replicate predicts with its own current
$w_k$. Lines are the across-replicate mean; bands are mean $\pm$ one sample standard
deviation (`ddof=1`) — repeat-run variability, not confidence or credible intervals.


## Colab bootstrap

The three modules are written next to the notebook so that `import anchored_lasso` works without cloning the repository.

In [ ]:
%%writefile anchored_sgld.py
"""Non-reversible anchored Langevin with block state-dependent skew-symmetric J.

Reusable library behind ``nonreversible_anchored_langevin.ipynb``.

The sampled object is the regression coefficient ``beta`` (written ``b`` in
code where it is a projection argument).  It is *unknown* and is what the
sampler explores; ``beta_true`` is the fixed vector used once to generate the
labels and is never used by the sampler.

Target
------
Uniform prior on a constraint set ``K`` times the logistic likelihood:

    pi_K(beta) ∝ exp(-U(beta)) 1_K(beta),
    U(beta) = sum_{j in train} [ log(1 + exp(X_j.beta)) - y_j X_j.beta ].

``U`` is a **sum** over the 1600 training rows, never a mean.  The mini-batch
estimator rescales accordingly,

    Ghat_k = (n_train / m) X_{B_k}^T [ sigmoid(X_{B_k} beta_k) - y_{B_k} ],

with ``B_k`` drawn uniformly without replacement within the iteration.

Anchor
------
    U0(beta) = U(beta) + rho H_K(beta),
    a(beta)  = exp(U(beta) - U0(beta)) = exp(-rho H_K(beta)).

``H_K`` is a purely geometric function of the constraint (``||beta||^2`` on the
ball, the normalised constraint value on the quartic set), so ``a`` is computed
**exactly from the geometry** — a noisy likelihood difference is never
exponentiated.  With ``rho = log 2`` and ``H_K in [0, 1]`` on ``K`` this gives
``1/2 <= a <= 1``.

This anchor is a *proposed non-trivial anchor for an already smooth target*,
used to make ``a`` state-dependent so the anchored machinery is exercised.  It
is **not** an extra Bayesian penalty: the target ``pi_K`` is unchanged, because
``a`` enters both the drift and the diffusion coefficient in the combination
that leaves ``exp(-U)`` invariant (see :func:`invariant_measure_note`).

Update
------
    beta_{k+1} = Pi_K[ beta_k - h a_k (v_k + alpha J(beta_k) v_k)
                       + sqrt(2 h a_k) xi_k ],
    v_k = Ghat_k + rho grad_H_K(beta_k),
    a_k = exp(-rho H_K(beta_k)),  xi_k ~ N(0, I_d).

No ``grad a`` correction is added, and ``J`` is never rescaled beyond the
constant block strengths ``s``.
"""

from __future__ import annotations

import math
import time
from dataclasses import dataclass, field, replace
from typing import Callable, Sequence

import numpy as np
from scipy.optimize import brentq
from scipy.special import expit
from sklearn.model_selection import train_test_split

RHO_ANCHORED: float = math.log(2.0)

BETA_TRUE_9 = np.array(
    [0.35, -0.25, 0.15, 0.30, -0.20, 0.10, 0.25, -0.15, 0.20]
)
BETA_TRUE_3 = np.array([0.60, -0.30, 0.20])


# ==========================================================================
# 1. Configuration
# ==========================================================================
@dataclass(frozen=True)
class ExperimentConfig:
    """Every knob of the experiment.  Nothing is hard-coded elsewhere."""

    d: int = 9
    n_total: int = 2000
    test_fraction: float = 0.2
    batch_size: int = 50                       # m
    n_iterations: int = 1000
    step_size: float = 1e-4                    # h (the paper's candidate value)
    n_repeats: int = 100                       # R
    block_scales: tuple[float, ...] = (10.0, 10.0, 10.0)
    epsilon: float = 0.2                       # smoothing of the l^p constraints
    Lambda: float = 1.0                        # quartic threshold
    l1_radius: float = 3.0                     # L1-smooth ball radius budget
    checkpoint_every: int = 10
    data_seed: int = 2026
    split_seed: int = 2027
    sampler_seed: int = 3000
    # Step-size sensitivity
    sensitivity_divisors: tuple[float, ...] = (1.0, 2.0, 4.0)
    sensitivity_repeats: int = 20
    # Reporting
    target_accuracy: float = 0.64

    @property
    def n_train(self) -> int:
        return self.n_total - int(round(self.n_total * self.test_fraction))

    @property
    def n_test(self) -> int:
        return int(round(self.n_total * self.test_fraction))

    @property
    def n_blocks(self) -> int:
        if self.d % 3 != 0:
            raise ValueError("d must be a multiple of 3 for the block construction")
        return self.d // 3

    @property
    def scales(self) -> np.ndarray:
        """Block strengths as an array of length ``n_blocks``."""
        s = np.asarray(self.block_scales, dtype=float)
        if s.size != self.n_blocks:
            raise ValueError(
                f"block_scales has {s.size} entries but d = {self.d} needs "
                f"{self.n_blocks}"
            )
        return s

    def beta_true(self) -> np.ndarray:
        if self.d == 9:
            return BETA_TRUE_9.copy()
        if self.d == 3:
            return BETA_TRUE_3.copy()
        raise ValueError("beta_true is specified only for d = 3 and d = 9")


CONFIG_D3 = ExperimentConfig(d=3, block_scales=(10.0,))


#: The four compared methods: (name, rho, alpha).
METHODS: tuple[tuple[str, float, float], ...] = (
    ("Projected SGLD", 0.0, 0.0),
    ("Non-reversible SGLD", 0.0, 1.0),
    ("Reversible anchored Langevin", RHO_ANCHORED, 0.0),
    ("Non-reversible anchored Langevin", RHO_ANCHORED, 1.0),
)

#: Colourblind-safe, validated (worst adjacent CVD deltaE 9.2 protan / 22.9
#: normal).  Line style is a deliberate second encoding channel.
METHOD_STYLE: dict[str, dict[str, object]] = {
    "Projected SGLD": {"color": "#0173B2", "ls": "-", "lw": 1.7},
    "Non-reversible SGLD": {"color": "#DE8F05", "ls": "--", "lw": 1.7},
    "Reversible anchored Langevin": {"color": "#029E73", "ls": "-.", "lw": 1.7},
    "Non-reversible anchored Langevin": {"color": "#CC3311", "ls": "-", "lw": 2.8},
}


def invariant_measure_note() -> str:
    """The continuous-time identity, and what it does *not* claim."""
    return (
        "Continuous process:  d.beta = -a(beta) [I + alpha J(beta)] grad_U0(beta) dt "
        "+ sqrt(2 a(beta)) dW.\n"
        "  * Reversible part.  For pi ∝ exp(-U0)/a the Fokker-Planck flux is\n"
        "    -b pi + grad(a pi) = a grad_U0 pi - grad_U0 (a pi) = 0, since a pi ∝ "
        "exp(-U0).\n"
        "    With a = exp(U - U0) this gives pi ∝ exp(-U0)/a = exp(-U): the anchor "
        "cancels exactly.\n"
        "  * Non-reversible part.  Its flux is -alpha a J grad_U0 pi = "
        "alpha J grad(exp(-U0)) x const,\n"
        "    whose divergence is (div J).grad f + trace(J Hess f) = 0 + 0, because "
        "div J = 0 and a\n"
        "    skew matrix has zero Frobenius inner product with a symmetric one. So "
        "pi is unchanged.\n"
        "\n"
        "WHAT THIS DOES NOT SAY.  The implemented algorithm is a *projected "
        "stochastic-gradient*\n"
        "Euler-Maruyama discretisation. Three separate sources of bias remain, none "
        "of which the\n"
        "identity above controls:\n"
        "  (i)   finite step size h (Euler-Maruyama discretisation bias, O(h) in "
        "general);\n"
        "  (ii)  mini-batch gradient noise, which injects extra variance not matched "
        "by the\n"
        "        sqrt(2 h a) term and inflates the effective temperature;\n"
        "  (iii) the projection Pi_K, which puts mass on the boundary and is not a "
        "discretisation\n"
        "        of any reflected process that preserves pi_K exactly.\n"
        "Accuracy curves therefore say nothing directly about posterior fidelity."
    )


# ==========================================================================
# 2. Synthetic data
# ==========================================================================
@dataclass
class Dataset:
    """Frozen data set and stratified split, shared by every method/geometry."""

    X_train: np.ndarray
    y_train: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    beta_true: np.ndarray
    data_seed: int
    split_seed: int

    @property
    def n_train(self) -> int:
        return self.X_train.shape[0]

    @property
    def n_test(self) -> int:
        return self.X_test.shape[0]

    @property
    def d(self) -> int:
        return self.X_train.shape[1]


def make_dataset(cfg: ExperimentConfig) -> Dataset:
    """Generate ``X_j ~ N(0, 2 I_d)`` and Bernoulli labels, then split 80/20.

    Coordinate standard deviation is ``sqrt(2)``.  There is no intercept and no
    feature standardisation.  Labels use the inverse-CDF form
    ``y_j = 1{u_j <= sigmoid(X_j.beta_true)}`` with ``u_j ~ Uniform(0,1)``.
    """
    rng = np.random.default_rng(cfg.data_seed)
    beta_true = cfg.beta_true()
    X = rng.normal(loc=0.0, scale=np.sqrt(2.0), size=(cfg.n_total, cfg.d))
    u = rng.uniform(0.0, 1.0, size=cfg.n_total)
    y = (u <= expit(X @ beta_true)).astype(float)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=cfg.test_fraction,
        random_state=cfg.split_seed,
        stratify=y,
        shuffle=True,
    )
    return Dataset(
        X_train=np.ascontiguousarray(X_train),
        y_train=np.ascontiguousarray(y_train),
        X_test=np.ascontiguousarray(X_test),
        y_test=np.ascontiguousarray(y_test),
        beta_true=beta_true,
        data_seed=cfg.data_seed,
        split_seed=cfg.split_seed,
    )


# ==========================================================================
# 3. Posterior and gradients  (beta may be (d,) or (R, d))
# ==========================================================================
def _as_2d(beta: np.ndarray) -> tuple[np.ndarray, bool]:
    beta = np.asarray(beta, dtype=float)
    if beta.ndim == 1:
        return beta[None, :], True
    return beta, False


def potential_U(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """``U(beta) = sum_j [softplus(X_j.beta) - y_j X_j.beta]`` (a SUM).

    ``np.logaddexp(0, z)`` is the numerically stable softplus; the
    label-dependent term ``- y_j X_j.beta`` is retained.
    """
    beta2, squeeze = _as_2d(beta)
    eta = beta2 @ X.T
    value = (np.logaddexp(0.0, eta) - y[None, :] * eta).sum(axis=1)
    return value[0] if squeeze else value


def full_gradient(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """``grad U = X^T (sigmoid(X beta) - y)`` over all rows of ``X``."""
    beta2, squeeze = _as_2d(beta)
    residual = expit(beta2 @ X.T) - y[None, :]
    gradient = residual @ X
    return gradient[0] if squeeze else gradient


def minibatch_gradient(
    beta: np.ndarray,
    X: np.ndarray,
    y: np.ndarray,
    batch_index: np.ndarray,
) -> np.ndarray:
    """``Ghat = (n_train/m) X_B^T [sigmoid(X_B beta) - y_B]``, vectorised over replicates.

    Parameters
    ----------
    beta
        ``(R, d)`` current states.
    X, y
        Full training arrays; ``n_train`` is taken from ``X``.
    batch_index
        ``(R, m)`` integer indices, drawn uniformly **without replacement**
        within each row.

    The ``n_train / m`` factor makes this an unbiased estimator of the *summed*
    gradient.  It is never replaced by an unscaled batch average, and never uses
    ``n_total``.
    """
    beta2, squeeze = _as_2d(beta)
    n_train, m = X.shape[0], batch_index.shape[-1]
    Xb = X[batch_index]                       # (R, m, d)
    yb = y[batch_index]                       # (R, m)
    eta = np.einsum("rmd,rd->rm", Xb, beta2)
    residual = expit(eta) - yb
    gradient = (n_train / m) * np.einsum("rmd,rm->rd", Xb, residual)
    return gradient[0] if squeeze else gradient


def accuracy(beta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Plug-in accuracy of ``1{sigmoid(X.beta) >= 0.5}``, vectorised over replicates."""
    beta2, squeeze = _as_2d(beta)
    prediction = (beta2 @ X.T) >= 0.0        # sigmoid(z) >= 0.5  <=>  z >= 0
    value = (prediction == (y[None, :] >= 0.5)).mean(axis=1)
    return value[0] if squeeze else value


# ==========================================================================
# 4. Geometries: constraint, anchor, projection, initialisation, J vectors
# ==========================================================================
class Geometry:
    """Interface shared by the ball and the quartic constraint set."""

    name: str
    d: int

    # --- constraint ---
    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    @property
    def threshold(self) -> float:
        raise NotImplementedError

    def feasible(self, beta: np.ndarray, tol: float = 1e-9) -> np.ndarray:
        return self.constraint_value(beta) <= self.threshold + tol

    # --- anchor ---
    def H(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    # --- geometry of the boundary ---
    def normal(self, beta: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    # --- block vectors defining J (before the block strengths) ---
    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``(R, n_blocks, 3)`` vectors ``w_l`` with block ``[s_l w_l]_x``."""
        raise NotImplementedError

    # --- projection and initialisation ---
    def project(self, z: np.ndarray) -> "ProjectionOutcome":
        raise NotImplementedError

    def sample_uniform(self, rng: np.random.Generator, n: int) -> np.ndarray:
        raise NotImplementedError


@dataclass
class ProjectionOutcome:
    """Result of projecting a batch of proposals."""

    beta: np.ndarray                 # (R, d) projected states
    projected: np.ndarray            # (R,) bool, was the row infeasible?
    max_kkt_residual: float          # worst stationarity residual over projected rows
    max_feasibility_excess: float    # worst g(b) - threshold after projection


class BallGeometry(Geometry):
    """``K = {beta : ||beta||_2^2 <= 1}``, with ``H_K(beta) = ||beta||^2``."""

    name = "unit ball"

    def __init__(self, d: int) -> None:
        self.d = d

    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        value = np.sum(beta2 * beta2, axis=1)
        return value[0] if squeeze else value

    @property
    def threshold(self) -> float:
        return 1.0

    def H(self, beta: np.ndarray) -> np.ndarray:
        return self.constraint_value(beta)

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        return 2.0 * np.asarray(beta, dtype=float)

    def normal(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        norm = np.linalg.norm(beta2, axis=1, keepdims=True)
        out = np.divide(beta2, norm, out=np.zeros_like(beta2), where=norm > 0)
        return out[0] if squeeze else out

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        direction = np.asarray(direction, dtype=float)
        return direction / np.linalg.norm(direction)

    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``w_l = beta_{I_l}`` — the ball's normal is proportional to ``beta``."""
        beta2, _ = _as_2d(beta)
        return beta2.reshape(beta2.shape[0], -1, 3)

    def project(self, z: np.ndarray) -> ProjectionOutcome:
        """Euclidean projection: ``z`` if ``||z|| <= 1``, else ``z/||z||``."""
        z2, squeeze = _as_2d(z)
        norm = np.linalg.norm(z2, axis=1)
        outside = norm > 1.0
        beta = z2.copy()
        if np.any(outside):
            beta[outside] = z2[outside] / norm[outside, None]
        # KKT: b(1 + 2 mu) = z with mu = (||z|| - 1)/2; residual is exactly 0.
        residual = 0.0
        if np.any(outside):
            mu = (norm[outside] - 1.0) / 2.0
            residual = float(
                np.abs(beta[outside] * (1.0 + 2.0 * mu[:, None]) - z2[outside]).max()
            )
        excess = float((np.sum(beta * beta, axis=1) - 1.0).max())
        out = beta[0] if squeeze else beta
        return ProjectionOutcome(out, outside, residual, excess)

    def sample_uniform(self, rng: np.random.Generator, n: int) -> np.ndarray:
        """Uniform on the ball: ``Z V^{1/d} / ||Z||``."""
        Z = rng.standard_normal((n, self.d))
        V = rng.random(n)
        radius = V ** (1.0 / self.d)
        return Z * (radius / np.linalg.norm(Z, axis=1))[:, None]


class QuarticGeometry(Geometry):
    """``K = {beta : g(beta) = sum_i (beta_i^2 + eps^2)^2 <= Lambda}``.

    ``g_min = d eps^4`` is attained at the origin, ``D = Lambda - d eps^4``, and

        H_K(beta) = (g(beta) - d eps^4) / D  in [0, 1] on K,
        grad_g[i] = 4 beta_i (beta_i^2 + eps^2),  grad_H_K = grad_g / D.
    """

    name = "quartic set"

    def __init__(self, d: int, epsilon: float = 0.2, Lambda: float = 1.0) -> None:
        self.d = d
        self.epsilon = epsilon
        self.Lambda = Lambda
        self.g_min = d * epsilon ** 4
        self.D = Lambda - self.g_min
        if self.D <= 0:
            raise ValueError("Lambda must exceed d * epsilon**4")

    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        value = np.sum((beta2 * beta2 + self.epsilon ** 2) ** 2, axis=1)
        return value[0] if squeeze else value

    @property
    def threshold(self) -> float:
        return self.Lambda

    def grad_g(self, beta: np.ndarray) -> np.ndarray:
        beta = np.asarray(beta, dtype=float)
        return 4.0 * beta * (beta * beta + self.epsilon ** 2)

    def H(self, beta: np.ndarray) -> np.ndarray:
        return (self.constraint_value(beta) - self.g_min) / self.D

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        return self.grad_g(beta) / self.D

    def normal(self, beta: np.ndarray) -> np.ndarray:
        gradient, squeeze = _as_2d(self.grad_g(beta))
        norm = np.linalg.norm(gradient, axis=1, keepdims=True)
        out = np.divide(gradient, norm, out=np.zeros_like(gradient), where=norm > 0)
        return out[0] if squeeze else out

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        direction = np.asarray(direction, dtype=float)
        objective = lambda t: float(self.constraint_value(t * direction)) - self.Lambda
        upper = 1.0
        while objective(upper) < 0.0:
            upper *= 2.0
        t = brentq(objective, 0.0, upper, xtol=1e-14, rtol=8.9e-16, maxiter=200)
        return t * direction

    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``w_l = -grad_{I_l} g(beta)``.

        The quartic normal is proportional to ``grad g``, not to ``beta``, so
        the ball's ``[s beta]_x`` block would generally violate ``J n = 0`` here.
        """
        gradient, _ = _as_2d(self.grad_g(beta))
        return (-gradient).reshape(gradient.shape[0], -1, 3)

    # ---- Euclidean projection via the KKT system ----
    def _solve_coordinates(self, z_abs: np.ndarray, mu) -> np.ndarray:
        """Solve ``b + 4 mu b (b^2 + eps^2) = |z|`` for ``b >= 0``, elementwise.

        The equation is the depressed cubic ``b^3 + p b + q = 0`` with
        ``p = (1 + 4 mu eps^2)/(4 mu) > 0`` and ``q = -|z|/(4 mu)``.  With
        ``p > 0`` there is exactly one real root, given stably by the hyperbolic
        form ``b = -2 sqrt(p/3) sinh( arcsinh( q / (2 (p/3)^{3/2}) ) / 3 )``.
        The map is strictly increasing in ``b`` (derivative
        ``1 + 12 mu b^2 + 4 mu eps^2 > 0``), so the root is unique.
        """
        mu_array = np.asarray(mu, dtype=float)
        scalar_mu = mu_array.ndim == 0
        if scalar_mu and mu_array <= 0.0:
            return z_abs.copy()
        safe = np.where(mu_array > 0.0, mu_array, 1.0)
        p = (1.0 + 4.0 * safe * self.epsilon ** 2) / (4.0 * safe)
        q = -z_abs / (4.0 * safe)
        scale = (p / 3.0) ** 1.5
        theta = np.arcsinh(q / (2.0 * scale)) / 3.0
        root = -2.0 * np.sqrt(p / 3.0) * np.sinh(theta)
        # mu = 0 leaves the point unchanged (the map is the identity there).
        return np.where(np.broadcast_to(mu_array > 0.0, root.shape), root, z_abs)

    def _project_one(self, z: np.ndarray) -> tuple[np.ndarray, float]:
        """Project a single infeasible point; returns ``(b, mu)``."""
        sign = np.sign(z)
        z_abs = np.abs(z)

        def gap(mu: float) -> float:
            b = self._solve_coordinates(z_abs, mu)
            return float(np.sum((b * b + self.epsilon ** 2) ** 2)) - self.Lambda

        # gap(0) = g(z) - Lambda > 0; gap is decreasing and tends to
        # d eps^4 - Lambda < 0, so a bracket always exists.
        mu_high = 1.0
        for _ in range(200):
            if gap(mu_high) <= 0.0:
                break
            mu_high *= 2.0
        else:  # pragma: no cover
            raise RuntimeError("failed to bracket the projection multiplier mu")
        mu = brentq(gap, 0.0, mu_high, xtol=1e-14, rtol=8.9e-16, maxiter=300)
        return sign * self._solve_coordinates(z_abs, mu), float(mu)

    def project(self, z: np.ndarray, n_bisect: int = 100) -> ProjectionOutcome:
        """Exact Euclidean projection onto ``{g <= Lambda}`` (never radial scaling).

        The outer multiplier ``mu`` is found by a bracketed bisection run on all
        infeasible rows simultaneously; ``mu -> g(b(mu))`` is monotonically
        decreasing, so bisection is unconditionally reliable, and ``n_bisect``
        halvings of the bracket reach machine precision.  :meth:`_project_one`
        keeps the scalar ``brentq`` version, which the checks use as an
        independent reference.
        """
        z2, squeeze = _as_2d(z)
        beta = z2.copy()
        outside = self.constraint_value(z2) > self.Lambda
        worst_kkt = 0.0

        rows = np.nonzero(outside)[0]
        if rows.size:
            z_rows = z2[rows]
            sign, z_abs = np.sign(z_rows), np.abs(z_rows)

            def gap(mu_column: np.ndarray) -> np.ndarray:
                b = self._solve_coordinates(z_abs, mu_column)
                return np.sum((b * b + self.epsilon ** 2) ** 2, axis=1) - self.Lambda

            # gap(0) > 0 by construction; grow the upper end until gap <= 0.
            low = np.zeros(rows.size)
            high = np.ones(rows.size)
            for _ in range(200):
                need = gap(high[:, None]) > 0.0
                if not need.any():
                    break
                high[need] *= 2.0
            else:  # pragma: no cover
                raise RuntimeError("failed to bracket the projection multiplier mu")

            for _ in range(n_bisect):
                mid = 0.5 * (low + high)
                positive = gap(mid[:, None]) > 0.0
                low = np.where(positive, mid, low)
                high = np.where(positive, high, mid)
            mu = 0.5 * (low + high)

            b = sign * self._solve_coordinates(z_abs, mu[:, None])
            beta[rows] = b
            worst_kkt = float(
                np.abs(b + mu[:, None] * self.grad_g(b) - z_rows).max()
            )
        excess = float((self.constraint_value(beta) - self.Lambda).max())
        out = beta[0] if squeeze else beta
        return ProjectionOutcome(out, outside, worst_kkt, excess)

    @property
    def box_half_width(self) -> float:
        """``b = sqrt( sqrt(Lambda - (d-1) eps^4) - eps^2 )``."""
        inner = math.sqrt(self.Lambda - (self.d - 1) * self.epsilon ** 4)
        return math.sqrt(inner - self.epsilon ** 2)

    def sample_uniform(
        self, rng: np.random.Generator, n: int, max_rounds: int = 10_000
    ) -> np.ndarray:
        """Uniform on ``K`` by rejection sampling from ``[-b, b]^d``."""
        half = self.box_half_width
        accepted: list[np.ndarray] = []
        total = 0
        kept = 0
        for _ in range(max_rounds):
            proposals = rng.uniform(-half, half, size=(max(n, 256), self.d))
            total += proposals.shape[0]
            good = proposals[self.constraint_value(proposals) <= self.Lambda]
            kept += good.shape[0]
            if good.size:
                accepted.append(good)
            if sum(a.shape[0] for a in accepted) >= n:
                break
        else:  # pragma: no cover
            raise RuntimeError("rejection sampler failed to fill the requested draws")
        out = np.vstack(accepted)[:n]
        self.last_acceptance_rate = kept / total
        return out


def make_geometry(name: str, cfg: ExperimentConfig) -> Geometry:
    if name == "ball":
        return BallGeometry(cfg.d)
    if name == "quartic":
        return QuarticGeometry(cfg.d, cfg.epsilon, cfg.Lambda)
    if name in ("l1smooth", "l1_smooth_ball"):
        return L1SmoothBallGeometry(cfg.d, cfg.epsilon, cfg.l1_radius)
    raise ValueError(f"unknown geometry {name!r}")


# ==========================================================================
# 5. Block state-dependent skew-symmetric matrix
# ==========================================================================
def apply_J(
    beta: np.ndarray,
    v: np.ndarray,
    geometry: Geometry,
    scales: np.ndarray,
) -> np.ndarray:
    """Matrix-free ``J(beta) v`` via block cross products.

    Each block acts as ``[s_l w_l]_x v_{I_l} = (s_l w_l) x v_{I_l}``, so no
    ``d x d`` matrix is ever formed.  The block strength multiplies ``w_l``
    exactly once.
    """
    beta2, squeeze = _as_2d(beta)
    v2, _ = _as_2d(v)
    w = geometry.j_block_vectors(beta2) * np.asarray(scales)[None, :, None]
    blocks = np.cross(w, v2.reshape(v2.shape[0], -1, 3))
    out = blocks.reshape(v2.shape[0], -1)
    return out[0] if squeeze else out


def hat(w: np.ndarray) -> np.ndarray:
    """``[w]_x``: the 3x3 cross-product matrix with ``[w]_x v = w x v``."""
    w1, w2, w3 = float(w[0]), float(w[1]), float(w[2])
    return np.array(
        [
            [0.0, -w3, w2],
            [w3, 0.0, -w1],
            [-w2, w1, 0.0],
        ]
    )


def build_J(beta: np.ndarray, geometry: Geometry, scales: np.ndarray) -> np.ndarray:
    """Explicit ``d x d`` matrix ``J(beta)`` — for verification, not for sampling."""
    beta = np.asarray(beta, dtype=float).ravel()
    d = beta.size
    w = geometry.j_block_vectors(beta[None, :])[0] * np.asarray(scales)[:, None]
    J = np.zeros((d, d))
    for block, vector in enumerate(w):
        lo = 3 * block
        J[lo : lo + 3, lo : lo + 3] = hat(vector)
    return J


def divergence_J(
    beta: np.ndarray, geometry: Geometry, scales: np.ndarray, step: float = 1e-5
) -> np.ndarray:
    """Centred finite-difference ``div(J)_i = sum_j dJ_ij/dbeta_j``."""
    beta = np.asarray(beta, dtype=float).ravel()
    divergence = np.zeros(beta.size)
    for j in range(beta.size):
        plus, minus = beta.copy(), beta.copy()
        plus[j] += step
        minus[j] -= step
        divergence += (
            build_J(plus, geometry, scales)[:, j]
            - build_J(minus, geometry, scales)[:, j]
        ) / (2.0 * step)
    return divergence


# ==========================================================================
# 6. Random streams (shared across the four methods)
# ==========================================================================
GEOMETRY_ID: dict[str, int] = {"unit ball": 11, "quartic set": 22,
                               "L1-smooth ball": 33}


@dataclass
class Streams:
    """Pre-generated randomness, identical for all four methods.

    Within one replicate and geometry every method sees the same starting
    coefficients, the same mini-batch index sequence and the same Gaussian
    increments; replicates use independent sub-streams.  Batch indices and
    Gaussian increments come from **separate** spawned streams, so the noise is
    independent of the current mini-batch.
    """

    beta_init: np.ndarray            # (R, d)
    batch_index: np.ndarray          # (R, n_iterations, m) int32
    noise: np.ndarray                # (R, n_iterations, d)
    seed_key: tuple[int, ...]

    @property
    def n_repeats(self) -> int:
        return self.beta_init.shape[0]

    @property
    def n_iterations(self) -> int:
        return self.noise.shape[1]


def make_streams(
    cfg: ExperimentConfig,
    geometry: Geometry,
    n_iterations: int,
    n_repeats: int,
    n_train: int,
) -> Streams:
    """Build the shared randomness for one geometry."""
    key = (cfg.sampler_seed, GEOMETRY_ID[geometry.name])
    base = np.random.SeedSequence(list(key))
    init_ss, batch_ss, noise_ss = base.spawn(3)

    beta_init = geometry.sample_uniform(np.random.default_rng(init_ss), n_repeats)

    batch_index = np.empty((n_repeats, n_iterations, cfg.batch_size), dtype=np.int32)
    for r, seed in enumerate(batch_ss.spawn(n_repeats)):
        rng = np.random.default_rng(seed)
        for k in range(n_iterations):
            batch_index[r, k] = rng.choice(n_train, size=cfg.batch_size, replace=False)

    noise = np.empty((n_repeats, n_iterations, cfg.d))
    for r, seed in enumerate(noise_ss.spawn(n_repeats)):
        noise[r] = np.random.default_rng(seed).standard_normal((n_iterations, cfg.d))

    return Streams(beta_init, batch_index, noise, key)


# ==========================================================================
# 7. The common sampler
# ==========================================================================
@dataclass
class RunResult:
    """Checkpointed output of one (method, geometry) run over ``R`` replicates."""

    method: str
    geometry: str
    rho: float
    alpha: float
    step_size: float
    n_iterations: int
    n_repeats: int
    block_scales: tuple[float, ...]
    checkpoints: np.ndarray          # (n_ckpt,) iteration indices
    train_accuracy: np.ndarray       # (n_ckpt, R)
    test_accuracy: np.ndarray        # (n_ckpt, R)
    beta: np.ndarray                 # (n_ckpt, R, d)
    train_loss: np.ndarray           # (n_ckpt, R)  full U on the training set
    constraint: np.ndarray           # (n_ckpt, R)  ||beta||^2 or g(beta)
    anchor: np.ndarray               # (n_ckpt, R)  a(beta)
    projection_rate: float
    max_kkt_residual: float
    max_feasibility_excess: float
    n_nonfinite: int
    runtime: float
    full_gradient: bool = False
    #: Mean over iterations and replicates of ||alpha J v|| / ||v||: how much
    #: larger the added non-reversible term is than the reversible drift.
    drift_ratio: float = 0.0
    #: Mean per-step displacement ||proposal - beta|| before projection.
    step_displacement: float = 0.0

    @property
    def simulated_time(self) -> np.ndarray:
        """``t = k h`` — the axis on which different step sizes are comparable."""
        return self.checkpoints * self.step_size

    def mean_std(self, which: str = "test_accuracy") -> tuple[np.ndarray, np.ndarray]:
        """Across-replicate mean and sample standard deviation (``ddof=1``)."""
        values = getattr(self, which)
        return values.mean(axis=1), values.std(axis=1, ddof=1)


def run_sampler(
    dataset: Dataset,
    geometry: Geometry,
    streams: Streams,
    *,
    method: str,
    rho: float,
    alpha: float,
    scales: np.ndarray,
    step_size: float,
    n_iterations: int,
    checkpoint_every: int,
    use_full_gradient: bool = False,
) -> RunResult:
    """One implementation; the four methods differ only in ``rho`` and ``alpha``.

        beta_{k+1} = Pi_K[ beta_k - h a_k (v_k + alpha J(beta_k) v_k)
                           + sqrt(2 h a_k) xi_k ]

    No ``grad a`` correction is added and ``J`` is never rescaled.
    """
    X_train, y_train = dataset.X_train, dataset.y_train
    X_test, y_test = dataset.X_test, dataset.y_test
    h = step_size
    beta = streams.beta_init.copy()
    n_repeats = beta.shape[0]

    checkpoints = [0]
    train_accuracy = [accuracy(beta, X_train, y_train)]
    test_accuracy = [accuracy(beta, X_test, y_test)]
    beta_history = [beta.copy()]
    train_loss = [potential_U(beta, X_train, y_train)]
    constraint = [geometry.constraint_value(beta)]
    anchor = [np.exp(-rho * geometry.H(beta))]

    n_projected = 0
    n_nonfinite = 0
    worst_kkt = 0.0
    worst_excess = -np.inf
    ratio_sum = 0.0
    displacement_sum = 0.0

    start = time.perf_counter()
    for k in range(n_iterations):
        if use_full_gradient:
            gradient = full_gradient(beta, X_train, y_train)
        else:
            gradient = minibatch_gradient(
                beta, X_train, y_train, streams.batch_index[:, k, :]
            )

        # v = Ghat + rho grad_H: the anchor derivative is added ONCE and is not
        # multiplied by n_train/m.
        v = gradient if rho == 0.0 else gradient + rho * geometry.grad_H(beta)
        a = np.ones(n_repeats) if rho == 0.0 else np.exp(-rho * geometry.H(beta))

        if alpha == 0.0:
            drift = v
        else:
            non_reversible = alpha * apply_J(beta, v, geometry, scales)
            drift = v + non_reversible
            # Diagnostic: how big is the added term relative to the reversible one?
            ratio_sum += float(
                np.mean(
                    np.linalg.norm(non_reversible, axis=1)
                    / np.maximum(np.linalg.norm(v, axis=1), 1e-300)
                )
            )
        proposal = (
            beta
            - h * a[:, None] * drift
            + np.sqrt(2.0 * h * a)[:, None] * streams.noise[:, k, :]
        )

        displacement_sum += float(np.mean(np.linalg.norm(proposal - beta, axis=1)))

        bad = ~np.isfinite(proposal).all(axis=1)
        if np.any(bad):
            n_nonfinite += int(bad.sum())
            proposal[bad] = beta[bad]          # freeze; counted and reported

        outcome = geometry.project(proposal)
        beta = outcome.beta
        n_projected += int(outcome.projected.sum())
        worst_kkt = max(worst_kkt, outcome.max_kkt_residual)
        worst_excess = max(worst_excess, outcome.max_feasibility_excess)

        if (k + 1) % checkpoint_every == 0:
            checkpoints.append(k + 1)
            train_accuracy.append(accuracy(beta, X_train, y_train))
            test_accuracy.append(accuracy(beta, X_test, y_test))
            beta_history.append(beta.copy())
            train_loss.append(potential_U(beta, X_train, y_train))
            constraint.append(geometry.constraint_value(beta))
            anchor.append(np.exp(-rho * geometry.H(beta)))
    runtime = time.perf_counter() - start

    return RunResult(
        method=method,
        geometry=geometry.name,
        rho=rho,
        alpha=alpha,
        step_size=h,
        n_iterations=n_iterations,
        n_repeats=n_repeats,
        block_scales=tuple(float(x) for x in np.atleast_1d(scales)),
        checkpoints=np.asarray(checkpoints),
        train_accuracy=np.asarray(train_accuracy),
        test_accuracy=np.asarray(test_accuracy),
        beta=np.asarray(beta_history),
        train_loss=np.asarray(train_loss),
        constraint=np.asarray(constraint),
        anchor=np.asarray(anchor),
        projection_rate=n_projected / (n_iterations * n_repeats),
        max_kkt_residual=worst_kkt,
        max_feasibility_excess=float(worst_excess),
        n_nonfinite=n_nonfinite,
        runtime=runtime,
        full_gradient=use_full_gradient,
        drift_ratio=ratio_sum / n_iterations,
        step_displacement=displacement_sum / n_iterations,
    )


def run_all_methods(
    dataset: Dataset,
    geometry: Geometry,
    cfg: ExperimentConfig,
    *,
    n_iterations: int | None = None,
    n_repeats: int | None = None,
    step_size: float | None = None,
    checkpoint_every: int | None = None,
    use_full_gradient: bool = False,
    verbose: bool = True,
) -> dict[str, RunResult]:
    """Run all four methods on one geometry with shared randomness."""
    n_iterations = n_iterations or cfg.n_iterations
    n_repeats = n_repeats or cfg.n_repeats
    step_size = cfg.step_size if step_size is None else step_size
    checkpoint_every = checkpoint_every or cfg.checkpoint_every

    streams = make_streams(cfg, geometry, n_iterations, n_repeats, dataset.n_train)
    results: dict[str, RunResult] = {}
    for name, rho, alpha in METHODS:
        result = run_sampler(
            dataset, geometry, streams,
            method=name, rho=rho, alpha=alpha, scales=cfg.scales,
            step_size=step_size, n_iterations=n_iterations,
            checkpoint_every=checkpoint_every, use_full_gradient=use_full_gradient,
        )
        results[name] = result
        if verbose:
            mean, sd = result.mean_std("test_accuracy")
            print(
                f"    {name:<34} final test acc {mean[-1]:.4f} +/- {sd[-1]:.4f}  "
                f"proj rate {result.projection_rate:.4f}  "
                f"nonfinite {result.n_nonfinite}  {result.runtime:5.1f}s",
                flush=True,
            )
    return results


# ==========================================================================
# 8. Mathematical implementation checks
# ==========================================================================
def check_gradient_finite_difference(
    dataset: Dataset, seed: int = 7, step: float = 1e-6
) -> dict[str, float]:
    """Compare ``full_gradient`` with centred finite differences of ``U``."""
    rng = np.random.default_rng(seed)
    beta = rng.normal(scale=0.3, size=dataset.d)
    analytic = full_gradient(beta, dataset.X_train, dataset.y_train)
    numerical = np.zeros_like(beta)
    for i in range(beta.size):
        plus, minus = beta.copy(), beta.copy()
        plus[i] += step
        minus[i] -= step
        numerical[i] = (
            potential_U(plus, dataset.X_train, dataset.y_train)
            - potential_U(minus, dataset.X_train, dataset.y_train)
        ) / (2.0 * step)
    absolute = float(np.abs(analytic - numerical).max())
    return {
        "max_absolute_error": absolute,
        "max_relative_error": absolute / float(np.abs(analytic).max()),
    }


def check_minibatch_scaling_exhaustive(
    n_toy: int = 8, m_toy: int = 3, d_toy: int = 3, seed: int = 11
) -> dict[str, float]:
    """Enumerate **every** batch of a toy data set and check unbiasedness.

    Averaging ``(n/m) X_B^T (sigmoid - y)`` over all ``C(n, m)`` subsets must
    reproduce the full summed gradient exactly.  This is what pins down the
    ``n_train / m`` factor: a plain batch average would be off by that factor.
    """
    from itertools import combinations

    rng = np.random.default_rng(seed)
    X = rng.normal(scale=np.sqrt(2.0), size=(n_toy, d_toy))
    y = (rng.random(n_toy) < 0.5).astype(float)
    beta = rng.normal(scale=0.4, size=d_toy)

    batches = np.array(list(combinations(range(n_toy), m_toy)), dtype=np.int32)
    estimates = minibatch_gradient(
        np.repeat(beta[None, :], batches.shape[0], axis=0), X, y, batches
    )
    exact = full_gradient(beta, X, y)
    averaged = estimates.mean(axis=0)

    # What an unscaled batch average would give, for contrast.
    unscaled = averaged * (m_toy / n_toy)
    return {
        "n_batches": float(batches.shape[0]),
        "max_abs_error_scaled": float(np.abs(averaged - exact).max()),
        "relative_error_scaled": float(
            np.abs(averaged - exact).max() / np.abs(exact).max()
        ),
        "max_abs_error_if_unscaled": float(np.abs(unscaled - exact).max()),
    }


def check_matrix_identities(
    geometry: Geometry, scales: np.ndarray, n_points: int = 12, seed: int = 13
) -> dict[str, float]:
    """Verify ``J^T = -J``, ``div J = 0``, ``J n = 0`` on the boundary, ``J grad_H = 0``."""
    rng = np.random.default_rng(seed)
    d = geometry.d
    interior = [np.zeros(d)] + [
        rng.normal(scale=0.35, size=d) for _ in range(n_points)
    ]
    boundary = [
        geometry.boundary_point(rng.normal(size=d)) for _ in range(n_points)
    ]

    skew = 0.0
    divergence = 0.0
    grad_h = 0.0
    matrix_free = 0.0
    for beta in interior + boundary:
        J = build_J(beta, geometry, scales)
        skew = max(skew, float(np.abs(J + J.T).max()))
        divergence = max(
            divergence, float(np.abs(divergence_J(beta, geometry, scales)).max())
        )
        grad_h = max(
            grad_h, float(np.abs(J @ geometry.grad_H(beta).ravel()).max())
        )
        v = rng.normal(size=d)
        matrix_free = max(
            matrix_free,
            float(np.abs(apply_J(beta, v, geometry, scales) - J @ v).max()),
        )

    tangency = 0.0
    boundary_error = 0.0
    for beta in boundary:
        J = build_J(beta, geometry, scales)
        normal = geometry.normal(beta).ravel()
        tangency = max(tangency, float(np.abs(J @ normal).max()))
        boundary_error = max(
            boundary_error,
            abs(float(geometry.constraint_value(beta)) - geometry.threshold),
        )
    return {
        "max_skew_error": skew,
        "max_divergence": divergence,
        "max_tangency_on_boundary": tangency,
        "max_J_gradH": grad_h,
        "max_matrix_free_vs_explicit": matrix_free,
        "max_boundary_residual": boundary_error,
    }


def check_ball_J_fails_on_quartic(
    cfg: ExperimentConfig, n_points: int = 8, seed: int = 17
) -> dict[str, float]:
    """Show that the **ball** matrix violates ``J n = 0`` on the quartic boundary.

    The ball's normal is parallel to ``beta``; the quartic normal is parallel to
    ``grad g``, whose coordinates are ``beta_i`` reweighted by
    ``4(beta_i^2 + eps^2)``.  Those directions differ unless every ``|beta_i|``
    is equal, so the unmodified block ``[s beta_I]_x`` is not tangential there.
    """
    rng = np.random.default_rng(seed)
    quartic = QuarticGeometry(cfg.d, cfg.epsilon, cfg.Lambda)
    ball = BallGeometry(cfg.d)
    correct = 0.0
    wrong = 0.0
    for _ in range(n_points):
        beta = quartic.boundary_point(rng.normal(size=cfg.d))
        normal = quartic.normal(beta).ravel()
        correct = max(
            correct,
            float(np.abs(build_J(beta, quartic, cfg.scales) @ normal).max()),
        )
        wrong = max(
            wrong, float(np.abs(build_J(beta, ball, cfg.scales) @ normal).max())
        )
    return {"quartic_J_tangency": correct, "ball_J_on_quartic_boundary": wrong}


def check_anchor_bounds(
    geometry: Geometry, rho: float = RHO_ANCHORED, n_points: int = 4000, seed: int = 19
) -> dict[str, float]:
    """On ``K``: ``H_K in [0, 1]`` and therefore ``1/2 <= a <= 1``."""
    rng = np.random.default_rng(seed)
    beta = geometry.sample_uniform(rng, n_points)
    H = geometry.H(beta)
    a = np.exp(-rho * H)
    return {
        "min_H": float(H.min()),
        "max_H": float(H.max()),
        "min_a": float(a.min()),
        "max_a": float(a.max()),
        "bounds_hold": float(
            bool(H.min() >= -1e-12 and H.max() <= 1 + 1e-12
                 and a.min() >= 0.5 - 1e-12 and a.max() <= 1 + 1e-12)
        ),
    }


def check_projection(
    geometry: Geometry, n_points: int = 40, scale: float = 1.6, seed: int = 23
) -> dict[str, float]:
    """Feasibility and KKT residuals, plus a radial-scaling comparison."""
    rng = np.random.default_rng(seed)
    z = rng.normal(scale=scale, size=(n_points, geometry.d))
    outcome = geometry.project(z)
    n_projected = int(outcome.projected.sum())

    # Radial scaling: t z with g(t z) = threshold. Feasible, but not the
    # Euclidean projection unless the set is a Euclidean ball.
    radial_worse = 0
    radial_gap = 0.0
    for row in np.nonzero(outcome.projected)[0]:
        objective = lambda t: (
            float(geometry.constraint_value(t * z[row])) - geometry.threshold
        )
        t = brentq(objective, 0.0, 1.0, xtol=1e-14, maxiter=200)
        d_radial = float(np.sum((t * z[row] - z[row]) ** 2))
        d_exact = float(np.sum((outcome.beta[row] - z[row]) ** 2))
        if d_radial > d_exact + 1e-12:
            radial_worse += 1
        radial_gap = max(radial_gap, d_radial - d_exact)
    return {
        "n_projected": float(n_projected),
        "max_kkt_residual": outcome.max_kkt_residual,
        "max_feasibility_excess": outcome.max_feasibility_excess,
        "n_radial_strictly_worse": float(radial_worse),
        "max_radial_excess_distance": radial_gap,
    }


def check_J_gradU0_nonzero(
    dataset: Dataset,
    geometry: Geometry,
    scales: np.ndarray,
    rho: float = RHO_ANCHORED,
    n_points: int = 8,
    seed: int = 29,
) -> dict[str, float]:
    """``J grad_H = 0`` exactly, but ``J grad_U0 = J grad_U`` must be nonzero.

    Otherwise the non-reversible term would do nothing at all.
    """
    rng = np.random.default_rng(seed)
    beta = geometry.sample_uniform(rng, n_points)
    grad_U0 = full_gradient(beta, dataset.X_train, dataset.y_train) + rho * geometry.grad_H(beta)
    J_grad = apply_J(beta, grad_U0, geometry, scales)
    norms = np.linalg.norm(J_grad, axis=1)
    return {
        "min_norm_J_gradU0": float(norms.min()),
        "median_norm_J_gradU0": float(np.median(norms)),
        "median_norm_gradU0": float(np.median(np.linalg.norm(grad_U0, axis=1))),
    }


def gradient_noise_report(
    dataset: Dataset,
    geometry: Geometry,
    cfg: ExperimentConfig,
    n_draws: int = 400,
    seed: int = 31,
) -> dict[str, float]:
    """Quantify how much ``J`` amplifies mini-batch gradient noise.

    The continuous-time invariance argument assumes the *exact* gradient.  With
    a stochastic gradient, ``J`` multiplies the gradient **error** as well as
    the gradient, so the added term injects extra variance that the
    ``sqrt(2 h a)`` term does not compensate.
    """
    rng = np.random.default_rng(seed)
    beta = geometry.sample_uniform(rng, 1)
    exact = full_gradient(beta, dataset.X_train, dataset.y_train)
    batches = np.stack(
        [rng.choice(dataset.n_train, cfg.batch_size, replace=False) for _ in range(n_draws)]
    )
    estimates = minibatch_gradient(
        np.repeat(beta, n_draws, axis=0), dataset.X_train, dataset.y_train, batches
    )
    error = np.linalg.norm(estimates - exact, axis=1).mean()

    J_exact = apply_J(beta, exact, geometry, cfg.scales)
    J_estimates = apply_J(
        np.repeat(beta, n_draws, axis=0), estimates, geometry, cfg.scales
    )
    J_error = np.linalg.norm(J_estimates - J_exact, axis=1).mean()

    h, a_typical = cfg.step_size, 0.75
    return {
        "norm_exact_gradient": float(np.linalg.norm(exact)),
        "mean_gradient_error": float(error),
        "mean_J_gradient_error": float(J_error),
        "amplification_factor": float(J_error / error),
        "step_from_gradient_noise": float(h * error),
        "step_from_J_gradient_noise": float(h * J_error),
        "injected_langevin_step": float(np.sqrt(2 * h * a_typical) * np.sqrt(cfg.d)),
    }


# ==========================================================================
# 9. Summaries
# ==========================================================================
def iterations_to_target(result: RunResult, target: float) -> np.ndarray:
    """First checkpoint iteration at which each replicate's TEST accuracy >= target.

    ``nan`` where a replicate never reaches the target.  Per replicate, never
    on the across-replicate mean.
    """
    reached = result.test_accuracy >= target              # (n_ckpt, R)
    out = np.full(result.n_repeats, np.nan)
    for r in range(result.n_repeats):
        hits = np.nonzero(reached[:, r])[0]
        if hits.size:
            out[r] = result.checkpoints[hits[0]]
    return out


def summarise(
    results: dict[str, RunResult], target: float
) -> "pd.DataFrame":  # noqa: F821
    """One row per method: final accuracy, time-to-target, projection, cost."""
    import pandas as pd

    rows = []
    for name, result in results.items():
        train_mean, train_sd = result.mean_std("train_accuracy")
        test_mean, test_sd = result.mean_std("test_accuracy")
        hits = iterations_to_target(result, target)
        reached = np.isfinite(hits)
        seconds_per_iteration_per_replicate = (
            result.runtime / result.n_iterations / result.n_repeats
        )
        rows.append(
            {
                "method": name,
                "rho": result.rho,
                "alpha": result.alpha,
                "final_train_acc": train_mean[-1],
                "final_train_sd": train_sd[-1],
                "final_test_acc": test_mean[-1],
                "final_test_sd": test_sd[-1],
                "frac_reaching_target": float(reached.mean()),
                "median_iters_to_target": (
                    float(np.median(hits[reached])) if reached.any() else np.nan
                ),
                "median_secs_to_target": (
                    float(np.median(hits[reached])) * seconds_per_iteration_per_replicate
                    if reached.any()
                    else np.nan
                ),
                "projection_rate": result.projection_rate,
                "drift_ratio_alphaJv_over_v": result.drift_ratio,
                "mean_step_displacement": result.step_displacement,
                "max_kkt_residual": result.max_kkt_residual,
                "n_nonfinite": result.n_nonfinite,
                "runtime_s": result.runtime,
                "secs_per_iter_per_replicate": seconds_per_iteration_per_replicate,
            }
        )
    return pd.DataFrame(rows)


def coefficient_moments(result: RunResult) -> dict[str, np.ndarray]:
    """Across-replicate mean and sd of each coordinate at every checkpoint."""
    return {
        "mean": result.beta.mean(axis=1),                  # (n_ckpt, d)
        "sd": result.beta.std(axis=1, ddof=1),             # (n_ckpt, d)
        "norm_mean": np.linalg.norm(result.beta, axis=2).mean(axis=1),
    }


# ==========================================================================
# 10. Figures
# ==========================================================================
def _stagger(values: Sequence[float], min_gap: float) -> list[float]:
    """Push overlapping label positions apart while preserving their order."""
    order = np.argsort(values)
    placed = np.asarray(values, dtype=float).copy()
    for rank in range(1, len(order)):
        lower, upper = order[rank - 1], order[rank]
        if placed[upper] - placed[lower] < min_gap:
            placed[upper] = placed[lower] + min_gap
    return list(placed)


def _draw_panel(ax, results: dict[str, RunResult], which: str, ylim, label_gap: float):
    finals = []
    for name, result in results.items():
        style = METHOD_STYLE[name]
        mean, sd = result.mean_std(which)
        lower = np.clip(mean - sd, 0.0, 1.0)      # clip the DISPLAYED band only
        upper = np.clip(mean + sd, 0.0, 1.0)
        ax.fill_between(
            result.checkpoints, lower, upper,
            color=style["color"], alpha=0.16, lw=0,
            zorder=2 if name.startswith("Non-reversible anchored") else 1,
        )
        ax.plot(
            result.checkpoints, mean,
            color=style["color"], ls=style["ls"], lw=style["lw"], label=name,
            zorder=4 if name.startswith("Non-reversible anchored") else 3,
        )
        finals.append((name, float(mean[-1]), result.checkpoints[-1]))

    positions = _stagger([value for _, value, _ in finals], label_gap)
    for (name, _, last), y in zip(finals, positions):
        ax.annotate(
            name.replace("Non-reversible", "Non-rev.").replace(" Langevin", ""),
            xy=(last, y), xytext=(6, 0), textcoords="offset points",
            color=METHOD_STYLE[name]["color"], fontsize=7.5, va="center",
            fontweight="bold" if name.startswith("Non-reversible anchored") else "normal",
        )
    ax.set_xlabel("Iterations")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.3)


def make_caption(cfg: ExperimentConfig, geometry_label: str, result: RunResult) -> str:
    """Caption carrying every setting the figure depends on."""
    return (
        f"d = {cfg.d};  constraint: {geometry_label};  step size h = {result.step_size:.3g};  "
        f"mini-batch m = {cfg.batch_size};  repeats R = {result.n_repeats};  "
        f"iterations = {result.n_iterations};  anchor rho = log 2 "
        f"({RHO_ANCHORED:.4f}) for anchored methods, 0 otherwise;  "
        f"block strengths s = {tuple(float(x) for x in cfg.scales)}.  "
        "Lines are the across-replicate mean; bands are mean +/- 1 sample sd "
        "(ddof = 1) of single-iterate accuracy, i.e. repeat-run variability -- "
        "NOT confidence or posterior credible intervals."
    )


def plot_accuracy_figure(
    results: dict[str, RunResult],
    cfg: ExperimentConfig,
    geometry_label: str,
    output_dir: str,
    tag: str,
    ylim: tuple[float, float] = (0.0, 1.0),
    title_extra: str = "",
) -> list[str]:
    """Training accuracy (left) and test accuracy (right) versus iteration."""
    import os

    import matplotlib.pyplot as plt

    any_result = next(iter(results.values()))
    zoomed = ylim != (0.0, 1.0)
    gap = (ylim[1] - ylim[0]) * 0.035

    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.4))
    _draw_panel(axes[0], results, "train_accuracy", ylim, gap)
    _draw_panel(axes[1], results, "test_accuracy", ylim, gap)
    axes[0].set_title(f"Training accuracy  (n = {cfg.n_train})", fontsize=11)
    axes[1].set_title(f"Test accuracy  (n = {cfg.n_test})", fontsize=11)
    axes[0].legend(fontsize=7.5, loc="lower right", framealpha=0.92)

    fig.suptitle(
        f"Non-reversible anchored Langevin — {geometry_label}, d = {cfg.d}{title_extra}",
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0.11, 1, 0.97))
    fig.text(
        0.5, 0.015, make_caption(cfg, geometry_label, any_result),
        ha="center", va="bottom", fontsize=7.4, wrap=True,
    )

    os.makedirs(output_dir, exist_ok=True)
    stem = os.path.join(output_dir, f"{tag}{'_zoom' if zoomed else ''}")
    paths = []
    for extension, kwargs in ((".png", {"dpi": 300}), (".pdf", {})):
        path = stem + extension
        fig.savefig(path, bbox_inches="tight", **kwargs)
        paths.append(path)
    plt.close(fig)
    return paths


def plot_sensitivity_figure(
    sensitivity: dict[float, dict[str, RunResult]],
    cfg: ExperimentConfig,
    geometry_label: str,
    output_dir: str,
    tag: str,
) -> list[str]:
    """Step-size sensitivity against **simulated time** ``t = k h``."""
    import os

    import matplotlib.pyplot as plt

    divisors = sorted(sensitivity)
    fig, axes = plt.subplots(2, len(divisors), figsize=(5.0 * len(divisors), 8.2),
                             squeeze=False)
    for column, divisor in enumerate(divisors):
        runs = sensitivity[divisor]
        for name, result in runs.items():
            style = METHOD_STYLE[name]
            mean, sd = result.mean_std("test_accuracy")
            t = result.simulated_time
            axes[0][column].fill_between(
                t, np.clip(mean - sd, 0, 1), np.clip(mean + sd, 0, 1),
                color=style["color"], alpha=0.15, lw=0,
            )
            axes[0][column].plot(
                t, mean, color=style["color"], ls=style["ls"], lw=style["lw"], label=name
            )
            axes[1][column].plot(
                result.simulated_time, result.constraint.mean(axis=1),
                color=style["color"], ls=style["ls"], lw=style["lw"], label=name,
            )
        step = next(iter(runs.values())).step_size
        iterations = next(iter(runs.values())).n_iterations
        axes[0][column].set_title(
            f"h/{int(divisor)} = {step:.3g},  {iterations} iterations", fontsize=10
        )
        axes[0][column].set_ylim(0.0, 1.0)
        axes[0][column].set_ylabel("Accuracy (test)")
        axes[1][column].axhline(
            next(iter(runs.values())).constraint.max() * 0 + _threshold_of(geometry_label, cfg),
            color="k", ls=":", lw=1.0, label="constraint threshold",
        )
        axes[1][column].set_ylabel("mean constraint value")
        for row in (0, 1):
            axes[row][column].set_xlabel("Simulated time  t = k h")
            axes[row][column].grid(alpha=0.3)
    axes[0][0].legend(fontsize=7, loc="lower right")
    axes[1][0].legend(fontsize=7, loc="upper right")
    fig.suptitle(
        f"Step-size sensitivity at constant simulated time — {geometry_label}, d = {cfg.d}",
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0.06, 1, 0.96))
    fig.text(
        0.5, 0.012,
        f"R = {next(iter(sensitivity[divisors[0]].values())).n_repeats} repeats per "
        f"step size (sensitivity setting);  m = {cfg.batch_size};  "
        f"s = {tuple(float(x) for x in cfg.scales)};  total simulated time held fixed by scaling the "
        "iteration count with 1/h.",
        ha="center", fontsize=7.4,
    )
    os.makedirs(output_dir, exist_ok=True)
    stem = os.path.join(output_dir, tag)
    paths = []
    for extension, kwargs in ((".png", {"dpi": 300}), (".pdf", {})):
        path = stem + extension
        fig.savefig(path, bbox_inches="tight", **kwargs)
        paths.append(path)
    plt.close(fig)
    return paths


def _threshold_of(geometry_label: str, cfg: ExperimentConfig) -> float:
    return 1.0 if "ball" in geometry_label else cfg.Lambda


# ==========================================================================
# 11. Persistence
# ==========================================================================
def save_results(
    path: str,
    results_by_geometry: dict[str, dict[str, RunResult]],
    cfg: ExperimentConfig,
    dataset: Dataset,
    extra: dict | None = None,
) -> str:
    """Save every checkpoint array plus metadata so figures are regenerable."""
    import json
    import os
    import platform

    import matplotlib
    import pandas as pd
    import scipy
    import sklearn

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    arrays: dict[str, np.ndarray] = {}
    meta_runs = []
    for geometry_label, results in results_by_geometry.items():
        for name, result in results.items():
            key = f"{geometry_label}|{name}".replace(" ", "_")
            arrays[f"{key}|checkpoints"] = result.checkpoints
            arrays[f"{key}|train_accuracy"] = result.train_accuracy
            arrays[f"{key}|test_accuracy"] = result.test_accuracy
            arrays[f"{key}|beta"] = result.beta
            arrays[f"{key}|train_loss"] = result.train_loss
            arrays[f"{key}|constraint"] = result.constraint
            arrays[f"{key}|anchor"] = result.anchor
            meta_runs.append(
                {
                    "geometry": geometry_label,
                    "method": name,
                    "rho": result.rho,
                    "alpha": result.alpha,
                    "step_size": result.step_size,
                    "n_iterations": result.n_iterations,
                    "n_repeats": result.n_repeats,
                    "block_scales": list(result.block_scales),
                    "projection_rate": result.projection_rate,
                    "max_kkt_residual": result.max_kkt_residual,
                    "max_feasibility_excess": result.max_feasibility_excess,
                    "n_nonfinite": result.n_nonfinite,
                    "runtime_s": result.runtime,
                    "drift_ratio": result.drift_ratio,
                    "step_displacement": result.step_displacement,
                    "full_gradient": result.full_gradient,
                }
            )
    np.savez_compressed(path, **arrays)

    metadata = {
        "config": {k: (list(v) if isinstance(v, tuple) else v)
                   for k, v in cfg.__dict__.items()},
        "derived": {
            "n_train": cfg.n_train, "n_test": cfg.n_test,
            "rho_anchored": RHO_ANCHORED, "n_blocks": cfg.n_blocks,
        },
        "seeds": {
            "data_seed": cfg.data_seed,
            "split_seed": cfg.split_seed,
            "sampler_seed": cfg.sampler_seed,
        },
        "beta_true": dataset.beta_true.tolist(),
        "runs": meta_runs,
        "versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "scipy": scipy.__version__,
            "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__,
            "matplotlib": matplotlib.__version__,
        },
    }
    if extra:
        metadata.update(extra)
    meta_path = os.path.splitext(path)[0] + "_metadata.json"
    with open(meta_path, "w") as handle:
        json.dump(metadata, handle, indent=2, default=str)
    return meta_path


def check_projection_solvers_agree(
    geometry: "QuarticGeometry", n_points: int = 25, scale: float = 1.6, seed: int = 37
) -> dict[str, float]:
    """Vectorised bisection (used in the sampler) vs scalar ``brentq`` reference."""
    rng = np.random.default_rng(seed)
    z = rng.normal(scale=scale, size=(n_points, geometry.d))
    fast = geometry.project(z).beta
    worst = 0.0
    for row in range(n_points):
        if geometry.constraint_value(z[row]) <= geometry.Lambda:
            reference = z[row]
        else:
            reference, _ = geometry._project_one(z[row])
        worst = max(worst, float(np.abs(fast[row] - reference).max()))
    return {"max_abs_difference": worst}


# ==========================================================================
# 12. Ill-conditioned design variant
# ==========================================================================
def make_correlated_dataset(cfg: ExperimentConfig, rho_x: float = 0.99) -> Dataset:
    """Same pipeline as :func:`make_dataset` but with AR(1)-correlated predictors.

    ``X_j ~ N(0, Sigma)`` with ``Sigma[i,k] = 2 * rho_x**|i-k|``.  The marginal
    coordinate variance is still 2, so only the *conditioning* changes: at
    ``rho_x = 0.99`` the posterior Hessian condition number is ~1200 rather than
    ~1.6 for the isotropic design.

    This exists because non-reversible perturbations buy their advantage from
    anisotropy.  With ``Sigma = 2I`` there is essentially nothing for ``J`` to
    exploit, which is a property of the *problem*, not of the sampler.
    """
    rng = np.random.default_rng(cfg.data_seed)
    index = np.arange(cfg.d)
    Sigma = 2.0 * (rho_x ** np.abs(index[:, None] - index[None, :]))
    X = rng.multivariate_normal(np.zeros(cfg.d), Sigma, size=cfg.n_total)
    beta_true = cfg.beta_true()
    y = (rng.uniform(size=cfg.n_total) <= expit(X @ beta_true)).astype(float)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_fraction, random_state=cfg.split_seed,
        stratify=y, shuffle=True,
    )
    return Dataset(
        X_train=np.ascontiguousarray(X_train), y_train=y_train,
        X_test=np.ascontiguousarray(X_test), y_test=y_test,
        beta_true=beta_true, data_seed=cfg.data_seed, split_seed=cfg.split_seed,
    )


def posterior_condition_number(dataset: Dataset) -> float:
    """Condition number of the observed-information matrix at ``beta_true``."""
    p = expit(dataset.X_train @ dataset.beta_true)
    weights = p * (1.0 - p)
    hessian = (dataset.X_train * weights[:, None]).T @ dataset.X_train
    eigenvalues = np.linalg.eigvalsh(hessian)
    return float(eigenvalues.max() / eigenvalues.min())


def paired_difference(a: np.ndarray, b: np.ndarray) -> tuple[float, float, float]:
    """Paired mean difference, its standard error and the t statistic.

    Valid because both methods run on identical starting points, mini-batch
    streams and Gaussian increments within each replicate, so the comparison is
    matched and the replicate-level noise cancels.
    """
    difference = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    mean = float(difference.mean())
    standard_error = float(difference.std(ddof=1) / np.sqrt(difference.size))
    return mean, standard_error, (mean / standard_error if standard_error > 0 else np.nan)


def ergodic_average(result: RunResult, burn_checkpoints: int) -> np.ndarray:
    """Time-averaged coefficient per replicate, ``(R, d)``.

    The asymptotic variance of such ergodic averages is exactly the quantity
    non-reversible perturbations are designed to reduce.
    """
    return result.beta[burn_checkpoints:].mean(axis=0)


# ==========================================================================
# 13. Smoothed L1 ("L1-smooth") ball
# ==========================================================================
class L1SmoothBallGeometry(Geometry):
    """``K = {beta : g(beta) = sum_i sqrt(beta_i^2 + eps^2) <= Lambda}``.

    The ``p = 1`` member of the same smoothed-l^p family as
    :class:`QuarticGeometry` (``p = 4``): a smooth surrogate for the L1 ball
    ``sum_i |beta_i| <= R``.  Following the same convention,

        g_min  = d * eps,          attained at the origin,
        Lambda = d * eps + R,      R the L1 radius budget,
        D      = Lambda - g_min = R,
        H_K    = (g(beta) - d*eps) / D  in [0, 1] on K,

        grad_g[i] = beta_i / sqrt(beta_i^2 + eps^2),   grad_H_K = grad_g / D.

    Note ``|grad_g[i]| < 1`` and it saturates towards 1 once ``|beta_i| >> eps``,
    so ``||J||`` is larger here than on the Euclidean ball at the same block
    strength: the useful range of ``s`` is correspondingly smaller.
    """

    name = "L1-smooth ball"

    def __init__(self, d: int, epsilon: float = 0.2, l1_radius: float = 3.0) -> None:
        self.d = d
        self.epsilon = epsilon
        self.l1_radius = l1_radius
        self.g_min = d * epsilon
        self.Lambda = self.g_min + l1_radius
        self.D = self.l1_radius
        if self.D <= 0:
            raise ValueError("l1_radius must be positive")

    # ---- constraint ----
    def constraint_value(self, beta: np.ndarray) -> np.ndarray:
        beta2, squeeze = _as_2d(beta)
        value = np.sqrt(beta2 * beta2 + self.epsilon ** 2).sum(axis=1)
        return value[0] if squeeze else value

    @property
    def threshold(self) -> float:
        return self.Lambda

    def grad_g(self, beta: np.ndarray) -> np.ndarray:
        beta = np.asarray(beta, dtype=float)
        return beta / np.sqrt(beta * beta + self.epsilon ** 2)

    # ---- anchor ----
    def H(self, beta: np.ndarray) -> np.ndarray:
        return (self.constraint_value(beta) - self.g_min) / self.D

    def grad_H(self, beta: np.ndarray) -> np.ndarray:
        return self.grad_g(beta) / self.D

    # ---- boundary geometry ----
    def normal(self, beta: np.ndarray) -> np.ndarray:
        gradient, squeeze = _as_2d(self.grad_g(beta))
        norm = np.linalg.norm(gradient, axis=1, keepdims=True)
        out = np.divide(gradient, norm, out=np.zeros_like(gradient), where=norm > 0)
        return out[0] if squeeze else out

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        direction = np.asarray(direction, dtype=float)
        objective = lambda t: float(self.constraint_value(t * direction)) - self.Lambda
        upper = 1.0
        while objective(upper) < 0.0:
            upper *= 2.0
        t = brentq(objective, 0.0, upper, xtol=1e-14, rtol=8.9e-16, maxiter=200)
        return t * direction

    def j_block_vectors(self, beta: np.ndarray) -> np.ndarray:
        """``w_l = -grad_{I_l} g(beta)`` — the normal here is parallel to grad g."""
        gradient, _ = _as_2d(self.grad_g(beta))
        return (-gradient).reshape(gradient.shape[0], -1, 3)

    # ---- Euclidean projection through the KKT system ----
    def _solve_coordinates(
        self, z_abs: np.ndarray, mu, inner_steps: int = 60
    ) -> np.ndarray:
        """Solve ``b + mu * b / sqrt(b^2 + eps^2) = |z|`` for ``b >= 0``.

        The map is odd and strictly increasing for ``mu >= 0`` — its derivative
        is ``1 + mu * eps^2 / (b^2 + eps^2)^{3/2} > 0`` — and satisfies
        ``phi(b) >= b``, so the root lies in ``[0, |z|]`` and bisection on that
        bracket is unconditionally reliable.  Unlike the quartic case there is no
        convenient closed form (the equation is quartic in ``b``), so the solve
        is a vectorised bisection over all rows and coordinates at once.
        """
        mu_array = np.asarray(mu, dtype=float)
        low = np.zeros_like(z_abs)
        high = z_abs.copy()
        for _ in range(inner_steps):
            mid = 0.5 * (low + high)
            value = mid + mu_array * mid / np.sqrt(mid * mid + self.epsilon ** 2)
            positive = value > z_abs
            high = np.where(positive, mid, high)
            low = np.where(positive, low, mid)
        return 0.5 * (low + high)

    def _project_one(self, z: np.ndarray) -> tuple[np.ndarray, float]:
        """Scalar-``brentq`` reference projection of a single infeasible point."""
        sign, z_abs = np.sign(z), np.abs(z)

        def gap(mu: float) -> float:
            b = self._solve_coordinates(z_abs, mu)
            return float(np.sqrt(b * b + self.epsilon ** 2).sum()) - self.Lambda

        mu_high = 1.0
        for _ in range(200):
            if gap(mu_high) <= 0.0:
                break
            mu_high *= 2.0
        else:  # pragma: no cover
            raise RuntimeError("failed to bracket the projection multiplier mu")
        mu = brentq(gap, 0.0, mu_high, xtol=1e-14, rtol=8.9e-16, maxiter=300)
        return sign * self._solve_coordinates(z_abs, mu), float(mu)

    def project(self, z: np.ndarray, n_bisect: int = 80) -> ProjectionOutcome:
        """Exact Euclidean projection onto ``K`` (never radial scaling)."""
        z2, squeeze = _as_2d(z)
        beta = z2.copy()
        outside = self.constraint_value(z2) > self.Lambda
        worst_kkt = 0.0

        rows = np.nonzero(outside)[0]
        if rows.size:
            z_rows = z2[rows]
            sign, z_abs = np.sign(z_rows), np.abs(z_rows)

            def gap(mu_column: np.ndarray) -> np.ndarray:
                b = self._solve_coordinates(z_abs, mu_column)
                return np.sqrt(b * b + self.epsilon ** 2).sum(axis=1) - self.Lambda

            low = np.zeros(rows.size)
            high = np.ones(rows.size)
            for _ in range(200):
                need = gap(high[:, None]) > 0.0
                if not need.any():
                    break
                high[need] *= 2.0
            else:  # pragma: no cover
                raise RuntimeError("failed to bracket the projection multiplier mu")

            for _ in range(n_bisect):
                mid = 0.5 * (low + high)
                positive = gap(mid[:, None]) > 0.0
                low = np.where(positive, mid, low)
                high = np.where(positive, high, mid)
            mu = 0.5 * (low + high)

            b = sign * self._solve_coordinates(z_abs, mu[:, None])
            beta[rows] = b
            worst_kkt = float(
                np.abs(b + mu[:, None] * self.grad_g(b) - z_rows).max()
            )
        excess = float((self.constraint_value(beta) - self.Lambda).max())
        out = beta[0] if squeeze else beta
        return ProjectionOutcome(out, outside, worst_kkt, excess)

    # ---- uniform initialisation ----
    def sample_uniform(
        self, rng: np.random.Generator, n: int, max_rounds: int = 10_000
    ) -> np.ndarray:
        """Uniform on ``K`` by rejection from the enclosing exact L1 ball.

        Because ``|x| <= sqrt(x^2 + eps^2) <= |x| + eps``,

            {||beta||_1 <= Lambda - d*eps}  subset  K  subset  {||beta||_1 <= Lambda},

        so proposals are drawn uniformly on the **exact** L1 ball of radius
        ``Lambda`` and kept when ``g(beta) <= Lambda``.  Box rejection, which
        works for the quartic set, is hopeless here: in nine dimensions an L1
        ball occupies about ``1e-6`` of its bounding box.

        Uniform draws on the exact L1 ball use the Barthe-Guedon-Mendelson-Naor
        construction: with ``g_i`` iid Laplace(0,1) and ``E ~ Exp(1)``,
        ``g / (||g||_1 + E)`` is uniform on the unit L1 ball.
        """
        accepted: list[np.ndarray] = []
        total = kept = 0
        for _ in range(max_rounds):
            size = max(n, 512)
            laplace = rng.laplace(0.0, 1.0, size=(size, self.d))
            exponential = rng.exponential(1.0, size=size)
            proposals = (
                self.Lambda
                * laplace
                / (np.abs(laplace).sum(axis=1) + exponential)[:, None]
            )
            total += size
            good = proposals[self.constraint_value(proposals) <= self.Lambda]
            kept += good.shape[0]
            if good.size:
                accepted.append(good)
            if sum(a.shape[0] for a in accepted) >= n:
                break
        else:  # pragma: no cover
            raise RuntimeError("rejection sampler failed to fill the requested draws")
        self.last_acceptance_rate = kept / total
        return np.vstack(accepted)[:n]


In [ ]:
%%writefile anchored_lasso.py
"""Exact-gradient anchored Langevin on a NON-differentiable LASSO target.

Target (the split is the user's):

    U(w) = sum_j [softplus(x_j.w) - y_j x_j.w] + w0^2/(2 sigma^2)   <- f, differentiable
           + lambda_lasso * sum_{j>=1} |w_j|                        <- g, NOT differentiable

so ``U = f + g`` with ``g`` kinked at every ``w_j = 0`` (the intercept ``w0`` is
excluded from the penalty and carries a weak Gaussian prior instead).

Anchor.  ``g`` is replaced by its smoothed surrogate

    g_delta(w) = lambda_lasso * sum_{j>=1} sqrt(w_j^2 + delta^2),
    U0 = f + g_delta,
    a(w) = exp(U - U0) = exp(lambda_lasso * sum_{j>=1} (|w_j| - sqrt(w_j^2 + delta^2))),

computed from the penalty difference directly, never as ``exp(U)/exp(U0)``.
Because ``0 <= sqrt(t^2+delta^2) - |t| <= delta``,

    exp(-(d-1) * lambda_lasso * delta) <= a(w) <= 1.

This is a *real* anchor: only ``grad U0`` is ever evaluated, yet the sampled
measure is ``exp(-U)`` with the true kinked ``g``.  (Contrast the earlier
``U0 = U + rho H_K`` construction, where ``U`` was already smooth and the anchor
was a proposed geometric one.)

Update (exact gradient, sign convention as specified):

    x_{k+1} = Pi_K[ x_k - eta a(x_k) grad_U0(x_k)
                        + eta alpha a(x_k) J_s(x_k) grad_U0(x_k)
                        + sqrt(2 eta a(x_k)) xi_{k+1} ]

i.e. the drift is ``-eta a (I - alpha J) grad_U0``.  Note the PLUS on the ``J``
term: since ``J`` enters linearly and is skew, this is the previous
``(I + alpha J)`` convention with the rotation reversed (equivalently
``s -> -s``), and the invariance argument is unchanged by the sign.

Constraints: the Euclidean ball and the smoothed l^p ball, both handled by exact
Euclidean projection.
"""

from __future__ import annotations

import hashlib
import math
import time
from dataclasses import dataclass, field

import numpy as np
from scipy.optimize import brentq
from scipy.special import expit
from sklearn.model_selection import train_test_split

from anchored_sgld import (  # verified machinery, reused unchanged
    BallGeometry,
    Geometry,
    ProjectionOutcome,
    _as_2d,
    apply_J,
    build_J,
    divergence_J,
    hat,
    paired_difference,
    posterior_condition_number,
)

BETA_TRUE_9 = np.array([0.35, -0.25, 0.15, 0.30, -0.20, 0.10, 0.25, -0.15, 0.20])


# ==========================================================================
# Configuration
# ==========================================================================
@dataclass(frozen=True)
class LassoConfig:
    """Settings for the exact-gradient LASSO experiment."""

    d: int = 9                       # 1 intercept + 8 slopes
    n_total: int = 2000
    test_fraction: float = 0.2
    # target
    lambda_lasso: float = 10.0       # strength of the non-differentiable g
    sigma_intercept: float = 5.0     # weak Gaussian prior on w0
    delta_anchor: float = 0.02       # smoothing of g inside U0
    # sampler
    eta: float = 1e-4                # step size
    n_iterations: int = 1000
    n_repeats: int = 100
    checkpoint_every: int = 10
    block_scales: tuple[float, ...] = (5.0, 5.0, 5.0)
    # l^p ball
    p_constraint: float = 1.0        # smoothed L1 ball
    epsilon: float = 0.2
    l1_radius: float = 1.9           # Lambda = d*eps + l1_radius = 3.7
    # design
    rho_x: float | None = None       # None -> X ~ N(0, 2I); else AR(1) correlation
    # seeds
    data_seed: int = 2026
    split_seed: int = 2027
    sampler_seed: int = 3000

    @property
    def n_train(self) -> int:
        return self.n_total - int(round(self.n_total * self.test_fraction))

    @property
    def n_test(self) -> int:
        return int(round(self.n_total * self.test_fraction))

    @property
    def n_blocks(self) -> int:
        if self.d % 3:
            raise ValueError("d must be a multiple of 3")
        return self.d // 3

    @property
    def scales(self) -> np.ndarray:
        s = np.asarray(self.block_scales, dtype=float)
        if s.size != self.n_blocks:
            raise ValueError(f"block_scales needs {self.n_blocks} entries")
        return s

    def beta_true(self) -> np.ndarray:
        if self.d != 9:
            raise ValueError("beta_true is specified for d = 9")
        return BETA_TRUE_9.copy()


#: (name, alpha) for the two compared methods; both anchored.
METHODS = (("Reversible anchored Langevin", 0.0),
           ("Non-reversible anchored Langevin", 1.0))


# ==========================================================================
# Data — now WITH an intercept column, because U penalises w0 separately
# ==========================================================================
@dataclass
class Dataset:
    X_train: np.ndarray
    y_train: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    beta_true: np.ndarray

    @property
    def n_train(self) -> int: return self.X_train.shape[0]
    @property
    def n_test(self) -> int: return self.X_test.shape[0]
    @property
    def d(self) -> int: return self.X_train.shape[1]


def make_dataset(cfg: LassoConfig) -> Dataset:
    """Synthetic logistic data with an intercept in column 0.

    ``U`` gives ``w0`` a Gaussian prior and penalises only ``w_1..w_8``, so
    column 0 is a column of ones and ``w0`` is a genuine intercept.  This is a
    deliberate change from the earlier no-intercept design, forced by the form
    of ``U``.
    """
    rng = np.random.default_rng(cfg.data_seed)
    n_features = cfg.d - 1
    if cfg.rho_x is None:
        Z = rng.normal(0.0, math.sqrt(2.0), size=(cfg.n_total, n_features))
    else:
        index = np.arange(n_features)
        Sigma = 2.0 * (cfg.rho_x ** np.abs(index[:, None] - index[None, :]))
        Z = rng.multivariate_normal(np.zeros(n_features), Sigma, size=cfg.n_total)
    X = np.hstack([np.ones((cfg.n_total, 1)), Z])
    beta_true = cfg.beta_true()
    y = (rng.uniform(size=cfg.n_total) <= expit(X @ beta_true)).astype(float)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_fraction, random_state=cfg.split_seed,
        stratify=y, shuffle=True,
    )
    return Dataset(np.ascontiguousarray(X_train), y_train,
                   np.ascontiguousarray(X_test), y_test, beta_true)


# ==========================================================================
# Target: U = f + g, anchor U0 = f + g_delta
# ==========================================================================
@dataclass
class LassoTarget:
    """``U = f + g`` with a non-differentiable ``g``, and its smooth anchor."""

    X: np.ndarray
    y: np.ndarray
    lambda_lasso: float
    sigma_intercept: float
    delta_anchor: float

    @property
    def d(self) -> int:
        return self.X.shape[1]

    # ---- f: the differentiable part ----
    def f(self, w: np.ndarray) -> np.ndarray:
        w2, squeeze = _as_2d(w)
        eta = w2 @ self.X.T
        value = (np.logaddexp(0.0, eta) - self.y[None, :] * eta).sum(axis=1)
        value = value + w2[:, 0] ** 2 / (2.0 * self.sigma_intercept ** 2)
        return value[0] if squeeze else value

    def grad_f(self, w: np.ndarray) -> np.ndarray:
        w2, squeeze = _as_2d(w)
        gradient = (expit(w2 @ self.X.T) - self.y[None, :]) @ self.X
        gradient = gradient.copy()
        gradient[:, 0] += w2[:, 0] / self.sigma_intercept ** 2
        return gradient[0] if squeeze else gradient

    # ---- g: the NON-differentiable part, and its smoothing ----
    def g(self, w: np.ndarray) -> np.ndarray:
        w2, squeeze = _as_2d(w)
        value = self.lambda_lasso * np.abs(w2[:, 1:]).sum(axis=1)
        return value[0] if squeeze else value

    def g_smooth(self, w: np.ndarray) -> np.ndarray:
        w2, squeeze = _as_2d(w)
        value = self.lambda_lasso * np.sqrt(
            w2[:, 1:] ** 2 + self.delta_anchor ** 2).sum(axis=1)
        return value[0] if squeeze else value

    def grad_g_smooth(self, w: np.ndarray) -> np.ndarray:
        w2, squeeze = _as_2d(w)
        out = np.zeros_like(w2)
        out[:, 1:] = (self.lambda_lasso * w2[:, 1:]
                      / np.sqrt(w2[:, 1:] ** 2 + self.delta_anchor ** 2))
        return out[0] if squeeze else out

    # ---- potentials ----
    def U(self, w: np.ndarray) -> np.ndarray:
        return self.f(w) + self.g(w)

    def U0(self, w: np.ndarray) -> np.ndarray:
        return self.f(w) + self.g_smooth(w)

    def grad_U0(self, w: np.ndarray) -> np.ndarray:
        """The ONLY gradient the sampler evaluates — exact, no mini-batch."""
        return self.grad_f(w) + self.grad_g_smooth(w)

    # ---- anchor coefficient ----
    def log_a(self, w: np.ndarray) -> np.ndarray:
        """``U - U0 = g - g_smooth``, from the penalty difference directly."""
        w2, squeeze = _as_2d(w)
        slopes = w2[:, 1:]
        value = self.lambda_lasso * (
            np.abs(slopes) - np.sqrt(slopes ** 2 + self.delta_anchor ** 2)
        ).sum(axis=1)
        return value[0] if squeeze else value

    def a(self, w: np.ndarray) -> np.ndarray:
        return np.exp(self.log_a(w))

    @property
    def log_a_lower_bound(self) -> float:
        return -(self.d - 1) * self.lambda_lasso * self.delta_anchor

    @property
    def a_lower_bound(self) -> float:
        return float(np.exp(self.log_a_lower_bound))

    def lipschitz_constant(self) -> float:
        """Upper bound on ``||Hess U0||``: data curvature plus anchor curvature."""
        eig_max = float(np.linalg.eigvalsh(self.X.T @ self.X).max())
        return 0.25 * eig_max + max(1.0 / self.sigma_intercept ** 2,
                                    self.lambda_lasso / self.delta_anchor)


def accuracy(w: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    w2, squeeze = _as_2d(w)
    value = (((w2 @ X.T) >= 0.0) == (y[None, :] >= 0.5)).mean(axis=1)
    return value[0] if squeeze else value


# ==========================================================================
# Smoothed l^p ball (general p >= 1; p = 1 and p = 4 reproduce the earlier sets)
# ==========================================================================
class SmoothLpBallGeometry(Geometry):
    """``K = {w : g_p(w) = sum_i (w_i^2 + eps^2)^{p/2} <= Lambda}``.

    ``Lambda = d eps^p + radius^p``, so ``p = 4`` with ``eps = 0.2`` and
    ``radius = 0.9964`` reproduces the quartic set (``Lambda = 1``) and ``p = 1``
    reproduces the L1-smooth ball.  ``grad_g[i] = p w_i (w_i^2+eps^2)^{p/2-1}``,
    and the J blocks use ``-s_l grad_{I_l} g`` since the normal is parallel to
    ``grad g``.
    """

    def __init__(self, d: int, p: float = 4.0, epsilon: float = 0.2,
                 radius: float | None = None, Lambda: float | None = None) -> None:
        if p < 1.0:
            raise ValueError("p < 1 breaks monotonicity of the projection map")
        self.d, self.p, self.epsilon = d, p, epsilon
        self.g_min = d * epsilon ** p
        if (radius is None) == (Lambda is None):
            raise ValueError("give exactly one of radius or Lambda")
        if Lambda is None:
            self.radius, self.Lambda = radius, self.g_min + radius ** p
        else:
            self.Lambda = Lambda
            self.radius = (Lambda - self.g_min) ** (1.0 / p)
        self.D = self.Lambda - self.g_min
        self.name = f"smoothed l^{p:g} ball"

    def constraint_value(self, w: np.ndarray) -> np.ndarray:
        w2, squeeze = _as_2d(w)
        value = ((w2 * w2 + self.epsilon ** 2) ** (self.p / 2.0)).sum(axis=1)
        return value[0] if squeeze else value

    @property
    def threshold(self) -> float:
        return self.Lambda

    def grad_g(self, w: np.ndarray) -> np.ndarray:
        w = np.asarray(w, dtype=float)
        return self.p * w * (w * w + self.epsilon ** 2) ** (self.p / 2.0 - 1.0)

    def H(self, w: np.ndarray) -> np.ndarray:
        return (self.constraint_value(w) - self.g_min) / self.D

    def grad_H(self, w: np.ndarray) -> np.ndarray:
        return self.grad_g(w) / self.D

    def normal(self, w: np.ndarray) -> np.ndarray:
        gradient, squeeze = _as_2d(self.grad_g(w))
        norm = np.linalg.norm(gradient, axis=1, keepdims=True)
        out = np.divide(gradient, norm, out=np.zeros_like(gradient), where=norm > 0)
        return out[0] if squeeze else out

    def boundary_point(self, direction: np.ndarray) -> np.ndarray:
        direction = np.asarray(direction, dtype=float)
        objective = lambda t: float(self.constraint_value(t * direction)) - self.Lambda
        upper = 1.0
        while objective(upper) < 0.0:
            upper *= 2.0
        return brentq(objective, 0.0, upper, xtol=1e-14, maxiter=200) * direction

    def j_block_vectors(self, w: np.ndarray) -> np.ndarray:
        gradient, _ = _as_2d(self.grad_g(w))
        return (-gradient).reshape(gradient.shape[0], -1, 3)

    # ---- projection ----
    def _solve_coordinates(self, z_abs: np.ndarray, mu, inner: int = 60) -> np.ndarray:
        """``b + mu p b (b^2+eps^2)^{p/2-1} = |z|``, ``b`` in ``[0, |z|]``.

        Strictly increasing for ``p >= 1`` (derivative
        ``1 + mu p (b^2+eps^2)^{p/2-2}[(p-1)b^2 + eps^2] > 0``), so bisection on
        that bracket is unconditionally reliable for any ``p``.
        """
        mu_array = np.asarray(mu, dtype=float)
        low, high = np.zeros_like(z_abs), z_abs.copy()
        for _ in range(inner):
            mid = 0.5 * (low + high)
            value = mid + mu_array * self.p * mid * (
                mid * mid + self.epsilon ** 2) ** (self.p / 2.0 - 1.0)
            positive = value > z_abs
            high = np.where(positive, mid, high)
            low = np.where(positive, low, mid)
        return 0.5 * (low + high)

    def _project_one(self, z: np.ndarray) -> tuple[np.ndarray, float]:
        sign, z_abs = np.sign(z), np.abs(z)

        def gap(mu: float) -> float:
            b = self._solve_coordinates(z_abs, mu)
            return float(((b * b + self.epsilon ** 2) ** (self.p / 2.0)).sum()) - self.Lambda

        high = 1.0
        while gap(high) > 0.0:
            high *= 2.0
        mu = brentq(gap, 0.0, high, xtol=1e-14, maxiter=300)
        return sign * self._solve_coordinates(z_abs, mu), float(mu)

    def project(self, z: np.ndarray, n_bisect: int = 80) -> ProjectionOutcome:
        z2, squeeze = _as_2d(z)
        beta = z2.copy()
        outside = self.constraint_value(z2) > self.Lambda
        worst = 0.0
        rows = np.nonzero(outside)[0]
        if rows.size:
            z_rows = z2[rows]
            sign, z_abs = np.sign(z_rows), np.abs(z_rows)

            def gap(mu_col: np.ndarray) -> np.ndarray:
                b = self._solve_coordinates(z_abs, mu_col)
                return ((b * b + self.epsilon ** 2) ** (self.p / 2.0)).sum(axis=1) - self.Lambda

            low, high = np.zeros(rows.size), np.ones(rows.size)
            for _ in range(200):
                need = gap(high[:, None]) > 0.0
                if not need.any():
                    break
                high[need] *= 2.0
            for _ in range(n_bisect):
                mid = 0.5 * (low + high)
                positive = gap(mid[:, None]) > 0.0
                low = np.where(positive, mid, low)
                high = np.where(positive, high, mid)
            mu = 0.5 * (low + high)
            b = sign * self._solve_coordinates(z_abs, mu[:, None])
            beta[rows] = b
            worst = float(np.abs(b + mu[:, None] * self.grad_g(b) - z_rows).max())
        excess = float((self.constraint_value(beta) - self.Lambda).max())
        return ProjectionOutcome(beta[0] if squeeze else beta, outside, worst, excess)

    def sample_uniform(self, rng: np.random.Generator, n: int,
                       max_rounds: int = 20_000) -> np.ndarray:
        """Uniform on ``K`` by rejection from the enclosing box."""
        half = math.sqrt(
            max((self.Lambda - (self.d - 1) * self.epsilon ** self.p) ** (2.0 / self.p)
                - self.epsilon ** 2, 0.0))
        accepted, total, kept = [], 0, 0
        for _ in range(max_rounds):
            size = max(n, 512)
            proposals = rng.uniform(-half, half, size=(size, self.d))
            total += size
            good = proposals[self.constraint_value(proposals) <= self.Lambda]
            kept += good.shape[0]
            if good.size:
                accepted.append(good)
            if sum(a.shape[0] for a in accepted) >= n:
                break
        else:  # pragma: no cover
            raise RuntimeError("rejection sampler failed")
        self.last_acceptance_rate = kept / total
        return np.vstack(accepted)[:n]


# ==========================================================================
# Sampler — exact gradient, no mini-batch anywhere
# ==========================================================================
@dataclass
class Streams:
    """Shared randomness: identical starts and increments for both methods."""

    w_init: np.ndarray               # (R, d)
    noise: np.ndarray                # (R, n_iterations, d)


def make_streams(cfg: LassoConfig, geometry: Geometry, seed_offset: int = 0) -> Streams:
    # NOTE: Python's built-in hash() on str is salted per process, so it must not
    # be used to derive a seed -- that silently breaks reproducibility between
    # runs.  A stable digest of the geometry name is used instead.
    tag = int.from_bytes(hashlib.sha256(geometry.name.encode()).digest()[:4], "big")
    base = np.random.SeedSequence([cfg.sampler_seed + seed_offset, tag])
    init_ss, noise_ss = base.spawn(2)
    w_init = geometry.sample_uniform(np.random.default_rng(init_ss), cfg.n_repeats)
    noise = np.empty((cfg.n_repeats, cfg.n_iterations, cfg.d))
    for r, seed in enumerate(noise_ss.spawn(cfg.n_repeats)):
        noise[r] = np.random.default_rng(seed).standard_normal((cfg.n_iterations, cfg.d))
    return Streams(w_init, noise)


@dataclass
class RunResult:
    method: str
    geometry: str
    alpha: float
    eta: float
    block_scales: tuple[float, ...]
    checkpoints: np.ndarray
    train_accuracy: np.ndarray       # (n_ckpt, R)
    test_accuracy: np.ndarray        # (n_ckpt, R)
    w: np.ndarray                    # (n_ckpt, R, d)
    U: np.ndarray                    # (n_ckpt, R) the TRUE non-smooth potential
    anchor: np.ndarray               # (n_ckpt, R) a(w)
    constraint: np.ndarray           # (n_ckpt, R)
    projection_rate: float
    max_kkt_residual: float
    n_nonfinite: int
    runtime: float
    drift_ratio: float = 0.0

    @property
    def n_repeats(self) -> int:
        return self.train_accuracy.shape[1]

    def mean_std(self, which: str = "test_accuracy") -> tuple[np.ndarray, np.ndarray]:
        values = getattr(self, which)
        return values.mean(axis=1), values.std(axis=1, ddof=1)


def run_chain(dataset: Dataset, target: LassoTarget, geometry: Geometry,
              streams: Streams, cfg: LassoConfig, *, method: str,
              alpha: float) -> RunResult:
    """x_{k+1} = Pi_K[ x_k - eta a grad_U0 + eta alpha a J grad_U0 + sqrt(2 eta a) xi ].

    ``grad_U0`` is the EXACT gradient of the smooth anchor; nothing is
    subsampled.  Only ``U0`` is differentiated, while the invariant measure of
    the underlying diffusion is ``exp(-U)`` with the true non-differentiable
    ``g``.
    """
    eta, scales = cfg.eta, cfg.scales
    w = streams.w_init.copy()
    checkpoints = [0]
    train_acc = [accuracy(w, dataset.X_train, dataset.y_train)]
    test_acc = [accuracy(w, dataset.X_test, dataset.y_test)]
    history = [w.copy()]
    potential = [target.U(w)]
    anchor = [target.a(w)]
    constraint = [geometry.constraint_value(w)]

    n_projected = n_nonfinite = 0
    worst_kkt = ratio_sum = 0.0

    start = time.perf_counter()
    for k in range(cfg.n_iterations):
        gradient = target.grad_U0(w)                 # exact
        a = target.a(w)
        if alpha == 0.0:
            drift = -eta * a[:, None] * gradient
        else:
            rotated = apply_J(w, gradient, geometry, scales)
            drift = -eta * a[:, None] * gradient + eta * alpha * a[:, None] * rotated
            ratio_sum += float(np.mean(
                np.linalg.norm(alpha * rotated, axis=1)
                / np.maximum(np.linalg.norm(gradient, axis=1), 1e-300)))
        proposal = w + drift + np.sqrt(2.0 * eta * a)[:, None] * streams.noise[:, k, :]

        bad = ~np.isfinite(proposal).all(axis=1)
        if np.any(bad):
            n_nonfinite += int(bad.sum())
            proposal[bad] = w[bad]

        outcome = geometry.project(proposal)
        w = outcome.beta
        n_projected += int(outcome.projected.sum())
        worst_kkt = max(worst_kkt, outcome.max_kkt_residual)

        if (k + 1) % cfg.checkpoint_every == 0:
            checkpoints.append(k + 1)
            train_acc.append(accuracy(w, dataset.X_train, dataset.y_train))
            test_acc.append(accuracy(w, dataset.X_test, dataset.y_test))
            history.append(w.copy())
            potential.append(target.U(w))
            anchor.append(target.a(w))
            constraint.append(geometry.constraint_value(w))
    runtime = time.perf_counter() - start

    return RunResult(
        method=method, geometry=geometry.name, alpha=alpha, eta=eta,
        block_scales=tuple(float(x) for x in np.atleast_1d(scales)),
        checkpoints=np.asarray(checkpoints),
        train_accuracy=np.asarray(train_acc), test_accuracy=np.asarray(test_acc),
        w=np.asarray(history), U=np.asarray(potential), anchor=np.asarray(anchor),
        constraint=np.asarray(constraint),
        projection_rate=n_projected / (cfg.n_iterations * cfg.n_repeats),
        max_kkt_residual=worst_kkt, n_nonfinite=n_nonfinite, runtime=runtime,
        drift_ratio=ratio_sum / cfg.n_iterations,
    )


def run_both(dataset: Dataset, target: LassoTarget, geometry: Geometry,
             cfg: LassoConfig, seed_offset: int = 0,
             verbose: bool = True) -> dict[str, RunResult]:
    """Both anchored methods on shared randomness (so comparisons are paired)."""
    streams = make_streams(cfg, geometry, seed_offset)
    results = {}
    for name, alpha in METHODS:
        results[name] = run_chain(dataset, target, geometry, streams, cfg,
                                  method=name, alpha=alpha)
        if verbose:
            run = results[name]
            mean, sd = run.mean_std("test_accuracy")
            print(f"    {name:<34} test {mean[-1]:.4f} +/- {sd[-1]:.4f}  "
                  f"proj {run.projection_rate:.3f}  ||aJg||/||g|| {run.drift_ratio:.2f}  "
                  f"nonfinite {run.n_nonfinite}  {run.runtime:5.1f}s", flush=True)
    return results


def make_block_anisotropic_dataset(
    cfg: LassoConfig,
    eigenvalues: tuple[float, ...] = (100.0, 1.0, 0.01),
    seed: int | None = None,
) -> Dataset:
    """Design whose anisotropy lives INSIDE each coordinate triple.

    ``Sigma_X`` is block diagonal on the slope coordinates, matching the block
    structure of ``J``, with each 3x3 block a random rotation of
    ``diag(eigenvalues)``.

    Why this matters.  ``J`` is built from the *constraint* geometry and is block
    diagonal on the triples ``(0,1,2), (3,4,5), (6,7,8)``, so it can only rotate
    *within* a triple.  A non-reversible perturbation accelerates convergence by
    coupling slow and fast directions of the target; if the target's slow
    directions span blocks (as under an AR(1) design) ``J`` cannot reach them.
    Putting the anisotropy inside the blocks aligns the two structures.  The
    linearised per-iteration rate ``-log rho(I - eta a (I - alpha J) H)`` rises
    from a 1.1x speed-up on the isotropic design and 2.2x under AR(1) to over 5x
    here.
    """
    rng = np.random.default_rng(cfg.data_seed if seed is None else seed)
    n_features = cfg.d - 1
    values = np.asarray(eigenvalues, dtype=float)
    Sigma = np.zeros((n_features, n_features))
    for start in range(0, n_features - n_features % 3, 3):
        rotation, _ = np.linalg.qr(rng.normal(size=(3, 3)))
        Sigma[start:start + 3, start:start + 3] = (
            rotation @ np.diag(values) @ rotation.T)
    remainder = n_features % 3
    if remainder:
        rotation, _ = np.linalg.qr(rng.normal(size=(remainder, remainder)))
        Sigma[-remainder:, -remainder:] = (
            rotation @ np.diag(values[:remainder]) @ rotation.T)

    Z = rng.multivariate_normal(np.zeros(n_features), Sigma, size=cfg.n_total)
    X = np.hstack([np.ones((cfg.n_total, 1)), Z])
    beta_true = cfg.beta_true()
    y = (rng.uniform(size=cfg.n_total) <= expit(X @ beta_true)).astype(float)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_fraction, random_state=cfg.split_seed,
        stratify=y, shuffle=True)
    return Dataset(np.ascontiguousarray(X_train), y_train,
                   np.ascontiguousarray(X_test), y_test, beta_true)


def make_aligned_design_dataset(
    cfg: LassoConfig,
    kappa: float = 100.0,
    base_variance: float = 2.0,
    seed: int | None = None,
) -> Dataset:
    """Design chosen from the theory of the block cross-product ``J``.

    Within a coordinate triple ``J_I = s [v]_x`` rotates in the plane
    perpendicular to ``v`` (``v = w*_I`` on the ball).  With block-Hessian
    eigenpairs ``(lambda_1 >= lambda_2 >= lambda_3)``, the rotation only helps the
    directions it touches: if the slow direction ``q_3`` is perpendicular to
    ``v`` its rate is replaced, for ``s|v| >= (lambda_2-lambda_3)/(2 sqrt(lambda_2
    lambda_3))``, by the arithmetic mean ``(lambda_2+lambda_3)/2`` -- a speed-up of
    ``(kappa+1)/2`` with ``kappa = lambda_2/lambda_3``.  If instead ``q_3`` is the
    rotation axis it is untouched and there is no gain.

    So the design puts, in every triple, ONE low-variance feature direction
    (variance ``base_variance/kappa``) perpendicular to ``beta_true_I``, and keeps
    the rest isotropic at ``base_variance``.  The slow posterior mode is then
    exactly the direction ``J`` rotates.  Keeping ``beta`` in the high-variance
    plane also keeps the linear predictor on the O(1) scale, so ``p(1-p)`` does not
    collapse and the Fisher curvature stays where the theory assumes it is.

    Block 1 holds the intercept (constant column), so its slow direction lives in
    the 2-d slope sub-block and is the unique direction perpendicular to
    ``(beta_1, beta_2)`` there.
    """
    rng = np.random.default_rng(cfg.data_seed if seed is None else seed)
    beta_true = cfg.beta_true()
    n_slopes = cfg.d - 1
    Sigma = base_variance * np.eye(n_slopes)

    def perpendicular(vector: np.ndarray) -> np.ndarray:
        """A deterministic unit vector perpendicular to ``vector``."""
        vector = vector / np.linalg.norm(vector)
        probe = np.zeros_like(vector)
        probe[int(np.argmin(np.abs(vector)))] = 1.0
        q = probe - (probe @ vector) * vector
        return q / np.linalg.norm(q)

    # Slope coordinates are 1..8 (0 is the intercept).  Blocks on ALL coordinates
    # are (0,1,2), (3,4,5), (6,7,8); in slope indexing: (0,1), (2,3,4), (5,6,7).
    blocks_in_slope_index = [(0, 1), (2, 3, 4), (5, 6, 7)]
    for block in blocks_in_slope_index:
        idx = np.asarray(block)
        beta_block = beta_true[idx + 1]
        q_slow = perpendicular(beta_block)
        Sigma[np.ix_(idx, idx)] -= (base_variance - base_variance / kappa) * np.outer(q_slow, q_slow)

    Z = rng.multivariate_normal(np.zeros(n_slopes), Sigma, size=cfg.n_total)
    X = np.hstack([np.ones((cfg.n_total, 1)), Z])
    y = (rng.uniform(size=cfg.n_total) <= expit(X @ beta_true)).astype(float)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_fraction, random_state=cfg.split_seed,
        stratify=y, shuffle=True)
    return Dataset(np.ascontiguousarray(X_train), y_train,
                   np.ascontiguousarray(X_test), y_test, beta_true)


def inplane_design(
    v_axis: float, v_fast: float, v_slow: float, b: float, e: float,
    b1: float = 0.3, block_1_variance: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Slope covariance and ``beta`` that put the slow direction where ``J`` rotates.

    In each slope triple ``(3,4,5)`` and ``(6,7,8)`` the covariance is
    ``v_axis q1 q1' + v_fast q2 q2' + v_slow q3 q3'`` with ``q1 = (1,1,1)/sqrt3``
    (the L1 rotation axis when the three coefficients are positive), ``q2 =
    (0,1,-1)/sqrt2`` (fast partner, ``q2 . beta = 0`` so it carries no signal)
    and ``q3 = (2,-1,-1)/sqrt6`` (slow, perpendicular to the axis, and
    signal-carrying because ``beta_triple = (b, e, e)`` with ``b > e``).  Block 1
    (intercept + slopes 1, 2) is isotropic with variance ``block_1_variance`` and
    coefficients ``(0, b1, b1)``.  Returns ``(Sigma_8x8, beta_9)``.
    """
    q1 = np.ones(3) / np.sqrt(3.0)
    q2 = np.array([0.0, 1.0, -1.0]) / np.sqrt(2.0)
    q3 = np.array([2.0, -1.0, -1.0]) / np.sqrt(6.0)
    block = v_axis * np.outer(q1, q1) + v_fast * np.outer(q2, q2) + v_slow * np.outer(q3, q3)
    Sigma = block_1_variance * np.eye(8)
    Sigma[2:5, 2:5] = block
    Sigma[5:8, 5:8] = block
    beta = np.array([0.0, b1, b1, b, e, e, b, e, e])
    return Sigma, beta


def make_scaled_dataset(
    cfg: LassoConfig,
    variances: tuple[float, ...],
    beta: tuple[float, ...],
    seed: int | None = None,
) -> Dataset:
    """Independent slope features with UNEQUAL scales and an explicit ``beta``.

    ``Z_j ~ N(0, variances[j])`` for the eight slopes (or ``Z ~ N(0, Sigma)``
    when ``variances`` is a full 8x8 covariance), a column of ones for the
    intercept, and ``beta`` the full 9-vector (intercept first).  This is the
    "unstandardised covariates" situation.  The Hessian of ``U0`` at the mode is
    ``X^T W X`` and inherits the feature scales, so a low-variance coordinate is a
    SLOW posterior direction that still carries whatever signal its coefficient
    gives it -- unlike the random-rotation and aligned designs, where the slow
    direction was either randomly placed or deliberately signal-free.

    The block cross-product ``J`` rotates within the triples ``(0,1,2), (3,4,5),
    (6,7,8)`` about the axis ``v_I = s w_I`` (ball) or ``v_I = -s grad g(w_I)``,
    i.e. a soft-sign of ``w_I`` (L1).  On the ball the axis at the mode is the
    mode itself, so the rotation never touches the direction of ``w*_I``; under
    the L1 geometry the axis is the (nearly) democratic sign vector, so every
    coordinate axis keeps a component of size ``sqrt(2/3)`` in the rotated plane.
    """
    rng = np.random.default_rng(cfg.data_seed if seed is None else seed)
    n_slopes = cfg.d - 1
    variances = np.asarray(variances, dtype=float)
    beta_true = np.asarray(beta, dtype=float)
    if beta_true.shape != (cfg.d,):
        raise ValueError("beta needs d entries")
    if variances.shape == (n_slopes,):
        Z = rng.normal(size=(cfg.n_total, n_slopes)) * np.sqrt(variances)
    elif variances.shape == (n_slopes, n_slopes):
        # a full slope covariance: lets a block's slow direction be any unit
        # vector (e.g. one perpendicular to the sign vector of beta_I)
        Z = rng.multivariate_normal(np.zeros(n_slopes), variances, size=cfg.n_total)
    else:
        raise ValueError("variances needs d-1 entries or a (d-1)x(d-1) covariance")
    X = np.hstack([np.ones((cfg.n_total, 1)), Z])
    y = (rng.uniform(size=cfg.n_total) <= expit(X @ beta_true)).astype(float)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=cfg.test_fraction, random_state=cfg.split_seed,
        stratify=y, shuffle=True)
    return Dataset(np.ascontiguousarray(X_train), y_train,
                   np.ascontiguousarray(X_test), y_test, beta_true)


def hessian_U0(target: LassoTarget, w: np.ndarray) -> np.ndarray:
    """Hessian of the smooth anchor at ``w`` (data curvature + priors + anchor)."""
    p = expit(target.X @ w)
    weights = p * (1.0 - p)
    H = (target.X * weights[:, None]).T @ target.X
    H[0, 0] += 1.0 / target.sigma_intercept ** 2
    slopes = np.arange(1, target.d)
    H[slopes, slopes] += (target.lambda_lasso * target.delta_anchor ** 2
                          / (w[1:] ** 2 + target.delta_anchor ** 2) ** 1.5)
    return H


def linearised_rates(target: LassoTarget, geometry: Geometry, w: np.ndarray,
                     scales: np.ndarray, eta: float) -> dict[str, float]:
    """Per-iteration convergence rates ``-log rho(I - eta a M)`` at the mode.

    ``M = H`` for the reversible chain and ``(I - J) H`` for the non-reversible
    one (sign convention of the sampler).  Returns both rates, the speed-up, and
    the implied number of iterations to relax by one e-fold.
    """
    H = hessian_U0(target, w)
    a = float(target.a(w))
    J = build_J(w, geometry, scales)
    d = H.shape[0]

    def rate(M):
        rho = np.abs(np.linalg.eigvals(np.eye(d) - eta * a * M)).max()
        return float(-np.log(rho)) if rho < 1.0 else float("-inf")

    r_rev, r_nr = rate(H), rate((np.eye(d) - J) @ H)
    eig = np.linalg.eigvalsh(H)
    return {"rate_rev": r_rev, "rate_nr": r_nr,
            "speed_up": (r_nr / r_rev if r_rev > 0 else float("nan")),
            "relax_rev": (1.0 / r_rev if r_rev > 0 else float("inf")),
            "relax_nr": (1.0 / r_nr if r_nr > 0 else float("inf")),
            "lambda_min": float(eig.min()), "lambda_max": float(eig.max()),
            "a_at_mode": a, "stable_rev": np.isfinite(r_rev), "stable_nr": np.isfinite(r_nr)}


In [ ]:
%%writefile nonreversible_beats_reversible.py
"""Non-reversible anchored Langevin beats the reversible one by ~0.2 in test accuracy.

The earlier win (``nonreversible_win.py``) was real but small (+0.004): the
random-rotation block-anisotropic design put the slow posterior direction at a
random angle to the axis of the block rotation, and the rotation only helps the
directions it sweeps.  The theory of the block cross-product ``J`` says exactly
where to put the anisotropy, and this script does it.

Within a coordinate triple ``I``, ``J_I = [v_I]_x`` rotates in the plane
PERPENDICULAR to its axis ``v_I`` (``v_I = s w_I`` on the ball, ``v_I = -s grad
g(w_I)`` ~ a soft-sign of ``w_I`` under the smoothed L1 ball).  Linearising the
drift at the mode, a slow Hessian eigen-direction ``q_3`` that lies IN that
plane, with fast partner ``q_2``, has its rate replaced by the arithmetic mean
``(lambda_2 + lambda_3)/2`` once ``sigma = s|v_I|`` exceeds ``sigma* =
(lambda_2 - lambda_3)/(2 sqrt(lambda_2 lambda_3))``: a speed-up of ``(kappa+1)/2``,
``kappa = lambda_2/lambda_3``.  A slow direction ALONG the axis is untouched, and
an oblique one gets only ``~1/cos^2(angle)``.  In discrete time the rotation is
stable while ``sigma^2 < (lambda_a + lambda_b)/(eta a lambda_a lambda_b) - 1``
(``lambda_a, lambda_b`` the Hessian restricted to the rotated plane).

So the design ("scaled" / unstandardised covariates) makes, inside each slope
triple, the covariance ``v_axis q1 q1' + v_fast q2 q2' + v_slow q3 q3'`` with
``q1 = (1,1,1)/sqrt3`` (the L1 axis when all three coefficients are positive),
``q2 = (0,1,-1)/sqrt2`` (fast, carries no signal: ``q2 . beta = 0``) and
``q3 = (2,-1,-1)/sqrt6`` (slow, IN the rotated plane, and carrying signal because
``beta_I = (b, e, e)`` with ``b >> e``).  The reversible chain then needs
``~1/(eta a lambda_3)`` iterations along ``q3`` while the non-reversible one needs
``~2/(eta a lambda_2)``.  The test-accuracy gap during that transient is large
because the residual along ``q3`` carries an O(1) share of the logit variance.

Both methods share initialisation, Gaussian increments and data; the ONLY
difference is ``alpha`` (0 vs 1).  The y-axis of the figure is NOT windowed.

    python nonreversible_beats_reversible.py [--quick]
"""

from __future__ import annotations

import argparse
import json
import os

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import anchored_lasso as lasso
import anchored_sgld as nral

OUTPUT_DIR = "results_beat"
LAMBDA, DELTA = 2.0, 0.02
STYLE = {"Reversible anchored Langevin":     {"color": "#0173B2", "ls": "--", "lw": 2.0},
         "Non-reversible anchored Langevin": {"color": "#CC3311", "ls": "-",  "lw": 2.6}}
Y_LIMITS = (0.40, 0.90)      # fixed, un-windowed: initialisation ~0.50, Bayes ceiling ~0.82

# The two headline configurations.  The ball has radius 1, so its beta is halved
# and its design variances quadrupled (identical logits, 4x the curvature).
CONFIGS = {
    "l1":   dict(v_axis=1.0, v_fast=64.0, v_slow=2.0, b=1.5, e=0.25, b1=0.3, block_1_variance=1.0,
                 s=4.0, eta=7e-6, epsilon=0.2, n_iterations=750, evaluate_at=100),
    "ball": dict(v_axis=4.0, v_fast=256.0, v_slow=4.0, b=0.5, e=0.125, b1=0.15, block_1_variance=4.0,
                 s=16.0, eta=2e-6, epsilon=0.2, n_iterations=750, evaluate_at=90),
}


def setup(tag: str, quick: bool):
    p = CONFIGS[tag]
    Sigma, beta = lasso.inplane_design(p["v_axis"], p["v_fast"], p["v_slow"], p["b"], p["e"],
                                       p["b1"], p["block_1_variance"])
    cfg = lasso.LassoConfig(
        lambda_lasso=LAMBDA, delta_anchor=DELTA, eta=p["eta"], block_scales=(p["s"],) * 3,
        epsilon=p["epsilon"], l1_radius=float(np.abs(beta).sum() + 1.0),
        n_repeats=20 if quick else 100,
        n_iterations=600 if quick else p["n_iterations"], checkpoint_every=10,
    )
    dataset = lasso.make_scaled_dataset(cfg, Sigma, beta)
    target = lasso.LassoTarget(dataset.X_train, dataset.y_train, cfg.lambda_lasso,
                               cfg.sigma_intercept, cfg.delta_anchor)
    geometry = (nral.BallGeometry(cfg.d) if tag == "ball"
                else nral.L1SmoothBallGeometry(cfg.d, cfg.epsilon, cfg.l1_radius))
    assert bool(geometry.feasible(dataset.beta_true)), f"beta_true outside {geometry.name}"
    return cfg, dataset, target, geometry


def figure(runs, cfg, geometry, tag, quick):
    p = CONFIGS[tag]
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.4))
    for ax, which, title in (
        (axes[0], "train_accuracy", f"Training accuracy  (n = {cfg.n_train})"),
        (axes[1], "test_accuracy",  f"Test accuracy  (n = {cfg.n_test})"),
    ):
        for name, run in runs.items():
            st = STYLE[name]; mean, sd = run.mean_std(which)
            ax.fill_between(run.checkpoints, np.clip(mean - sd, 0, 1),
                            np.clip(mean + sd, 0, 1), color=st["color"], alpha=0.15, lw=0)
            ax.plot(run.checkpoints, mean, color=st["color"], ls=st["ls"], lw=st["lw"], label=name)
        ax.set_ylim(*Y_LIMITS)
        ax.set_xlabel("Iterations"); ax.set_ylabel("Accuracy")
        ax.set_title(title, fontsize=11); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=9, loc="lower right")
    fig.suptitle(f"Non-reversible beats reversible anchored Langevin — {geometry.name}"
                 + ("  [QUICK MODE]" if quick else ""), fontsize=13)
    fig.tight_layout(rect=(0, 0.15, 1, 0.95))
    fig.text(0.5, 0.015,
        f"d = {cfg.d} (intercept + 8 slopes);  constraint: {geometry.name};  U = f + g with "
        f"g = {cfg.lambda_lasso:g}*sum|w_j| (non-differentiable), anchor delta = {cfg.delta_anchor};  "
        f"EXACT gradient;  eta = {cfg.eta:.1e};  s = {p['s']:g};  R = {cfg.n_repeats};  alpha = 0 vs 1.\n"
        f"Design: unstandardised covariates -- in each slope triple the covariance is "
        f"{p['v_axis']:g} q1q1' + {p['v_fast']:g} q2q2' + {p['v_slow']:g} q3q3' with q1 = (1,1,1)/sqrt3 "
        f"(the rotation axis), q2 = (0,1,-1)/sqrt2 (fast, no signal), q3 = (2,-1,-1)/sqrt6 (slow, in the "
        f"rotated plane, carries signal);  beta_triple = ({p['b']:g}, {p['e']:g}, {p['e']:g}).\n"
        "Lines: across-replicate mean; bands: mean +/- 1 sample sd of single-iterate accuracy "
        "(repeat-run variability, not confidence intervals).  Shared initialisation and noise "
        f"per replicate.  y-axis fixed to {Y_LIMITS} -- NOT windowed on the plateau.",
        ha="center", fontsize=7.2)
    paths = []
    for ext, kw in ((".png", {"dpi": 300}), (".pdf", {})):
        path = os.path.join(OUTPUT_DIR, f"accuracy_{tag}{ext}")
        fig.savefig(path, bbox_inches="tight", **kw); paths.append(path)
    plt.close(fig)
    return paths


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--quick", action="store_true")
    parser.add_argument("--geometry", choices=["l1", "ball", "both"], default="both")
    args = parser.parse_args()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tags = ["l1", "ball"] if args.geometry == "both" else [args.geometry]

    trajectory, held, curves, summary, figure_paths = [], [], [], {}, []
    offsets = (101,) if args.quick else (101, 202, 303, 404)
    for tag in tags:
        cfg, dataset, target, geometry = setup(tag, args.quick)
        p = CONFIGS[tag]
        ceiling = float(np.mean(np.maximum(nral.expit(dataset.X_test @ dataset.beta_true),
                                           1 - nral.expit(dataset.X_test @ dataset.beta_true))))
        print(f"\n=== {geometry.name}:  s {p['s']:g}, eta {cfg.eta:.1e}, R {cfg.n_repeats}, "
              f"{cfg.n_iterations} iterations, Bayes ceiling on the test set {ceiling:.4f}")
        runs = lasso.run_both(dataset, target, geometry, cfg, verbose=False)
        rev = runs["Reversible anchored Langevin"]; nr = runs["Non-reversible anchored Langevin"]
        print(f"   projection rate NR {nr.projection_rate:.4f}, REV {rev.projection_rate:.4f}; "
              f"non-finite {nr.n_nonfinite + rev.n_nonfinite}")
        for k, it in enumerate(rev.checkpoints):
            m, se, t = lasso.paired_difference(nr.test_accuracy[k], rev.test_accuracy[k])
            curves.append({"geometry": geometry.name, "iteration": int(it),
                           "rev_train": rev.train_accuracy[k].mean(), "nr_train": nr.train_accuracy[k].mean(),
                           "rev_test": rev.test_accuracy[k].mean(), "nr_test": nr.test_accuracy[k].mean(),
                           "rev_test_sd": rev.test_accuracy[k].std(ddof=1), "nr_test_sd": nr.test_accuracy[k].std(ddof=1),
                           "paired_diff": m, "paired_se": se, "t_stat": t})
        for it in sorted({k for k in (50, 100, 150, 200, 300, 400, 600, 1000, 1500) if k < cfg.n_iterations} | {cfg.n_iterations}):
            i = int(np.argmin(np.abs(rev.checkpoints - it)))
            m, se, t = lasso.paired_difference(nr.test_accuracy[i], rev.test_accuracy[i])
            trajectory.append({"geometry": geometry.name, "iteration": int(rev.checkpoints[i]),
                               "reversible": rev.test_accuracy[i].mean(),
                               "non_reversible": nr.test_accuracy[i].mean(),
                               "paired_diff": m, "paired_se": se, "t_stat": t})
            print(f"   it {int(rev.checkpoints[i]):4d}: REV {rev.test_accuracy[i].mean():.4f}  "
                  f"NR {nr.test_accuracy[i].mean():.4f}   diff {m:+.4f}  t {t:+6.2f}", flush=True)
        figure_paths += figure(runs, cfg, geometry, tag, args.quick)

        evaluate_at = p["evaluate_at"]
        print(f"   held-out confirmation at iteration {evaluate_at}:")
        diffs, ses = [], []
        for offset in offsets:
            runs2 = lasso.run_both(dataset, target, geometry, cfg, seed_offset=offset, verbose=False)
            r2 = runs2["Reversible anchored Langevin"]; n2 = runs2["Non-reversible anchored Langevin"]
            i = int(np.argmin(np.abs(r2.checkpoints - evaluate_at)))
            m, se, t = lasso.paired_difference(n2.test_accuracy[i], r2.test_accuracy[i])
            diffs.append(m); ses.append(se)
            held.append({"geometry": geometry.name, "seed_offset": offset, "iteration": int(r2.checkpoints[i]),
                         "reversible": r2.test_accuracy[i].mean(), "non_reversible": n2.test_accuracy[i].mean(),
                         "paired_diff": m, "paired_se": se, "t_stat": t})
            print(f"      offset {offset}: REV {r2.test_accuracy[i].mean():.4f}  NR {n2.test_accuracy[i].mean():.4f}"
                  f"  diff {m:+.4f}  t {t:+6.2f}", flush=True)
        pooled_se = float(np.sqrt(np.sum(np.array(ses) ** 2)) / len(ses)); pooled = float(np.mean(diffs))
        summary[geometry.name] = {
            "evaluate_at": evaluate_at, "pooled_diff": pooled, "pooled_se": pooled_se,
            "pooled_t": pooled / pooled_se, "all_positive": bool(all(d > 0 for d in diffs)),
            "confirmed": bool(all(d > 0 for d in diffs) and pooled / pooled_se > 2),
            "projection_rate_nr": nr.projection_rate, "bayes_ceiling_test": ceiling,
            "config": {**p, "lambda_lasso": LAMBDA, "delta_anchor": DELTA, "l1_radius": cfg.l1_radius,
                       "n_repeats": cfg.n_repeats, "n_iterations": cfg.n_iterations},
        }
        print(f"      POOLED {pooled:+.4f} (SE {pooled_se:.4f}, t = {pooled/pooled_se:+.2f}); all positive: "
              f"{summary[geometry.name]['all_positive']}  ->  "
              f"{'CONFIRMED' if summary[geometry.name]['confirmed'] else 'NOT confirmed'}")

    pd.DataFrame(curves).to_csv(os.path.join(OUTPUT_DIR, "curves.csv"), index=False)
    pd.DataFrame(trajectory).to_csv(os.path.join(OUTPUT_DIR, "trajectory.csv"), index=False)
    pd.DataFrame(held).to_csv(os.path.join(OUTPUT_DIR, "held_out.csv"), index=False)
    summary["figures"] = figure_paths
    with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as handle:
        json.dump(summary, handle, indent=2)
    print("\nwrote:", *sorted(os.listdir(OUTPUT_DIR)), sep="\n  ")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
import os, sys, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, os.path.abspath("."))
import anchored_lasso as lasso
import anchored_sgld as nral
import nonreversible_beats_reversible as beat   # sets the Agg backend; figures are shown with display(fig)

pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
QUICK  = os.environ.get("NRAL_QUICK", "0") == "1"
MODE   = "QUICK MODE" if QUICK else "FULL"
HELD_OUT = (101,) if QUICK else (101, 202, 303, 404)
OUT = "results_beat"; os.makedirs(OUT, exist_ok=True)
print(MODE)
print(pd.DataFrame(beat.CONFIGS).T)

## Data, target and the two constraint sets

The ball has radius 1, so its `beta` is halved and its design variances quadrupled relative to the $L_1$ configuration: identical logits, four times the curvature, hence the smaller `eta`.

In [ ]:
setups = {tag: beat.setup(tag, QUICK) for tag in ("l1", "ball")}
for tag, (cfg, dataset, target, geometry) in setups.items():
    z = dataset.X_test @ dataset.beta_true
    ceiling = float(np.mean(np.maximum(nral.expit(z), 1 - nral.expit(z))))
    print(f"{geometry.name:<16} feasible(beta_true) {bool(geometry.feasible(dataset.beta_true))}   "
          f"|beta|_1 {np.abs(dataset.beta_true).sum():.2f}  |beta|_2 {np.linalg.norm(dataset.beta_true):.2f}   "
          f"Bayes ceiling on the test set {ceiling:.4f}   a in [{target.a_lower_bound:.3f}, 1]   "
          f"eta*L = {cfg.eta * target.lipschitz_constant():.3f}   R = {cfg.n_repeats}, {cfg.n_iterations} iterations")

## Run both methods on both constraint sets

In [ ]:
results = {}
for tag, (cfg, dataset, target, geometry) in setups.items():
    t0 = time.time(); print(f"[{MODE}] {geometry.name}", flush=True)
    results[tag] = lasso.run_both(dataset, target, geometry, cfg, verbose=False)
    nr = results[tag]["Non-reversible anchored Langevin"]
    print(f"   {time.time() - t0:.0f} s;  projection rate NR {nr.projection_rate:.4f};  non-finite {nr.n_nonfinite}")

## Figures — training and test accuracy only

Vertical axis fixed to $[0.4, 0.9]$ on every panel.

In [ ]:
STYLE = {"Reversible anchored Langevin":     {"color": "#0173B2", "ls": "--", "lw": 2.0},
         "Non-reversible anchored Langevin": {"color": "#CC3311", "ls": "-",  "lw": 2.6}}
Y_LIMITS = (0.40, 0.90)          # fixed for every panel: NOT windowed on the plateau

for tag, (cfg, dataset, target, geometry) in setups.items():
    runs = results[tag]; p = beat.CONFIGS[tag]
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2))
    for ax, which, title in ((axes[0], "train_accuracy", f"Training accuracy  (n = {cfg.n_train})"),
                             (axes[1], "test_accuracy",  f"Test accuracy  (n = {cfg.n_test})")):
        for name, run in runs.items():
            st = STYLE[name]; mean, sd = run.mean_std(which)
            ax.fill_between(run.checkpoints, np.clip(mean - sd, 0, 1), np.clip(mean + sd, 0, 1),
                            color=st["color"], alpha=0.15, lw=0)
            ax.plot(run.checkpoints, mean, color=st["color"], ls=st["ls"], lw=st["lw"], label=name)
        ax.set_ylim(*Y_LIMITS); ax.set_xlabel("Iterations"); ax.set_ylabel("Accuracy")
        ax.set_title(title, fontsize=11); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=9, loc="lower right")
    fig.suptitle(f"Non-reversible beats reversible anchored Langevin — {geometry.name}  [{MODE}]", fontsize=13)
    fig.tight_layout(rect=(0, 0.10, 1, 0.95))
    fig.text(0.5, 0.012,
             f"s = {p['s']:g}, eta = {cfg.eta:.1e}, R = {cfg.n_repeats}, lambda = {cfg.lambda_lasso:g}, delta = {cfg.delta_anchor}; "
             f"design: per slope triple {p['v_axis']:g} q1q1' + {p['v_fast']:g} q2q2' + {p['v_slow']:g} q3q3', "
             f"beta_triple = ({p['b']:g}, {p['e']:g}, {p['e']:g});  y-axis fixed to {Y_LIMITS}, not windowed.",
             ha="center", fontsize=7.5)
    png = os.path.join(OUT, f"notebook_accuracy_{tag}.png")
    fig.savefig(png, dpi=150, bbox_inches="tight"); plt.close(fig)
    display(Image(filename=png))          # backend-independent: the saved figure is shown inline

## The numbers

Paired per-replicate differences of single-iterate test accuracy (both methods share starting points and Gaussian increments within a replicate).

In [ ]:
rows = []
for tag, (cfg, dataset, target, geometry) in setups.items():
    rev = results[tag]["Reversible anchored Langevin"]; nr = results[tag]["Non-reversible anchored Langevin"]
    for k in sorted({k for k in (50, 100, 150, 200, 400, 600) if k < cfg.n_iterations} | {cfg.n_iterations}):
        i = int(np.argmin(np.abs(rev.checkpoints - k)))
        m, se, t = lasso.paired_difference(nr.test_accuracy[i], rev.test_accuracy[i])
        rows.append({"geometry": geometry.name, "iteration": int(rev.checkpoints[i]),
                     "reversible": rev.test_accuracy[i].mean(), "non_reversible": nr.test_accuracy[i].mean(),
                     "paired_diff": m, "paired_se": se, "t_stat": t})
paired_table = pd.DataFrame(rows); display(paired_table)

### Held-out sampler seeds

The configuration is re-run on seeds that took no part in the search, at the iteration where the search-seed gap peaks.

In [ ]:
confirm, verdicts = [], {}
for tag, (cfg, dataset, target, geometry) in setups.items():
    evaluate_at = beat.CONFIGS[tag]["evaluate_at"]; diffs, ses = [], []
    for off in (0,) + HELD_OUT:
        runs = results[tag] if off == 0 else lasso.run_both(dataset, target, geometry, cfg, seed_offset=off, verbose=False)
        rev = runs["Reversible anchored Langevin"]; nr = runs["Non-reversible anchored Langevin"]
        i = int(np.argmin(np.abs(rev.checkpoints - evaluate_at)))
        m, se, t = lasso.paired_difference(nr.test_accuracy[i], rev.test_accuracy[i])
        confirm.append({"geometry": geometry.name, "seed_offset": off, "held_out": off != 0, "iteration": int(rev.checkpoints[i]),
                        "reversible": rev.test_accuracy[i].mean(), "non_reversible": nr.test_accuracy[i].mean(),
                        "paired_diff": m, "paired_se": se, "t_stat": t})
        if off != 0: diffs.append(m); ses.append(se)
    pooled = float(np.mean(diffs)); pooled_se = float(np.sqrt(np.sum(np.square(ses))) / len(ses))
    verdicts[geometry.name] = {"iteration": evaluate_at, "pooled_diff": pooled, "pooled_t": pooled / pooled_se,
                               "all_positive": all(d > 0 for d in diffs),
                               "confirmed": all(d > 0 for d in diffs) and pooled / pooled_se > 2}
confirm = pd.DataFrame(confirm); display(confirm)
for name, v in verdicts.items():
    print(f"{name:<16} held-out pooled diff {v['pooled_diff']:+.4f} at iteration {v['iteration']} (t = {v['pooled_t']:+.2f}), "
          f"all positive: {v['all_positive']}  ->  {'CONFIRMED' if v['confirmed'] else 'NOT confirmed'}")

## Result

**The non-reversible method beats the reversible one by a wide margin during the
transient, on both constraint sets, and the margin survives held-out sampler seeds.**
On the $L_1$ ball the test-accuracy gap at iteration 100 is about $+0.16$ (reversible
$\approx0.56$, non-reversible $\approx0.72$; pooled held-out $t\approx29$ with $R=100$ per
seed); on the unit ball it is about $+0.13$ at iteration 90. Both chains reach the same
plateau afterwards, and the paired difference at the end of the run is zero within noise —
the effect is faster convergence, not a different stationary accuracy.

## Where the win lives — the parameter range

From the one-factor-at-a-time map in `results_beat/parameter_map.md` (all with the $L_1$
geometry unless stated; "wins" means peak gap $\ge0.03$ with $t\ge3$):

| factor | wins | ties | loses / unstable |
|---|---|---|---|
| anisotropy $\kappa=v_{\rm fast}/v_{\rm slow}$ | $\kappa\ge4$; the gap grows with $\kappa$ and saturates at $\kappa\gtrsim32$ ($+0.18$) | $\kappa\le2$ | — |
| block strength $s$ | $0.5\le s\le12$ (gap $+0.03\to+0.21$) | $s=0$ | $s\ge16$: the projection fires on a third of the steps, late deficit |
| step size $\eta$ | $2\cdot10^{-6}\le\eta\le6\cdot10^{-5}$; the peak sits at $\approx0.7/(\eta\,a\,\lambda_{\rm slow})$ iterations | — | $\eta=10^{-4}$: $\eta a\lambda_{\max}>2$, unstable |
| $\lambda_{\rm lasso}$, $\delta$ | every value tried ($\lambda\in[0,30]$, $\delta\in[0.002,0.1]$) | — | — |
| sample size $n$ | every value tried ($250$–$8000$); the peak iteration scales as $1/n$ | — | — |
| initialisation | uniform on $K$ ($+0.18$), antipodal ($+0.65$) | origin ($+0.01$: the residual is then a pure shrinkage of the signal, which barely changes predictions) | — |
| slow-direction coefficient $b$ | $b\ge0.5$; the gap tracks the logit variance carried by $q_3$ | $b=0.25$ | — |
| geometry | $L_1$ (axis $\approx$ soft-sign, $q_3$ in the rotated plane); ball needs $\kappa\ge64$ and $s\ge8$ with $\eta\le5\cdot10^{-6}$ ($+0.10$ to $+0.13$) | — | ball with the slow coordinate along $w^*_I$ (axis theorem) |
| where the slow direction points | in the plane $\perp$ the rotation axis (this design) | isotropic design ($\kappa=1$); slow direction signal-free (`aligned`) | slow direction on a coordinate axis, oblique to the axis: $\le+0.06$, with a late deficit for $s\ge4$ |
| evaluation iteration | $0.15\lesssim\eta a\lambda_{\rm slow}\,k\lesssim1.5$ | later: both converged | — |
